# SQL интерфейс

Импорт библиотек

In [1]:
import polars as pl

Polars позволяет выполнять SQL-запросы прямо над *DataFrame* и *LazyFrame*, не подключая внешнюю базу данных. Это особенно удобно, если привыкли к синтаксису SQL и хотим быстро фильтровать, объединять и агрегировать данные с помощью знакомого языка.

Для этого в Polars есть два основных способа:
1. Функция верхнего уровня `pl.sql()` — простой способ выполнить один запрос.
2. Класс **SQLContext** — более гибкий инструмент для управления несколькими таблицами, повторного использования контекста и работы в блоках кода.

Здесь мы сосредоточимся над SQLContext: основной и наиболее мощный способ работы с SQL в Polars.

**SQLContext** — это специальный объект в Polars, который:
- регистрирует *DataFrame*/*LazyFrame* как «таблицы» с заданными именами,
- выполняет SQL-запросы над ними,
- управляет состоянием: можно временно зарегистрировать таблицы и автоматически удалить их после завершения блока кода.

**Замечание.** SQL в Polars — это обёртка над ленивыми вычислениями. Даже если на выходе получаем *DataFrame*, сам запрос сначала строится как *LazyFrame*, а затем выполняется. Это означает, что все оптимизации Polars (predicate pushdown, projection pushdown и т.д.) работают и в SQL-режиме. 

## SQLContext 

In [2]:
# Создаём данные
df = pl.DataFrame({"a": [1, 2, 3], "b": ["x", None, "z"]})

# Создаём контекст и регистрируем таблицу с именем "frame"
ctx = pl.SQLContext(frame=df)

# Выполняем SQL-запрос
result = ctx.execute("""
    SELECT b, a * 2 AS two_a 
    FROM frame
    WHERE b IS NOT NULL
""")

# Получаем результат (LazyFrame → DataFrame)
print(result.collect())

shape: (2, 2)
┌─────┬───────┐
│ b   ┆ two_a │
│ --- ┆ ---   │
│ str ┆ i64   │
╞═════╪═══════╡
│ x   ┆ 2     │
│ z   ┆ 6     │
└─────┴───────┘


Разбор кода:
1. Создаём данные: `df = pl.DataFrame({"a": [1, 2, 3], "b": ["x", None, "z"]})`;
2. Создаём контекст и регистрируем таблицу с именем "frame": `ctx = pl.SQLContext(frame=df)`;
3. Пишем запрос извлечения данных `ctx.execute`, в котором:
   - указываем, какие столбцы хотим видеть в результате: `SELECT b, a * 2 AS two_a`, где даём наименование `two_a` полю `a * 2`,
   - указываем, из какой таблицы берём данные: `FROM frame`,
   - фильтруем необходимые строки: `WHERE b IS NOT NULL`;
4. Выводим результат на экран: `print(result.collect())`.

Параметры конструктора **SQLContext**
```python
pl.SQLContext(
    frames=None,            # словарь {"имя": фрейм}
    register_globals=False, # автоматически регистрировать переменные из globals()
    eager=False,            # возвращать сразу DataFrame (иначе — LazyFrame)
    **named_frames          # именованные аргументы: table1=df1, table2=lf2
)
```

Основные параметры:
- `frames` - словарь вида {"таблица1": df1, "таблица2": lf2}. Поддерживаются:
  - Polars DataFrame и LazyFrame,
  - Pandas DataFrame и Series,
  - PyArrow Table и RecordBatch.
- `register_globals` - по умолчанию False. Если True, Polars автоматически найдёт все совместимые объекты в глобальной области видимости и зарегистрирует их под именами переменных. Например, если есть `df_sales = pl.DataFrame(...)`, то в SQL сможем писать `FROM df_sales`.
- `eager` - по умолчанию False. Если True, метод `.execute()` сразу вернёт *DataFrame*, а не *LazyFrame*. Удобно для быстрого просмотра результата.
- `**named_frames` - способ зарегистрировать сразу несколько таблиц, указав их имена и содержимое прямо в скобках при создании контекста. Например, `ctx = pl.SQLContext(sales=df1, customers=df2)`

С **SQLContext** можно работать в блоке `with`. Это полезно, чтобы автоматически очищать зарегистрированные таблицы после завершения блока:

In [3]:
df1 = pl.DataFrame({"id": [1, 2, 3], "value": [0.1, 0.2, 0.3]})
df2 = pl.DataFrame({"id": [3, 2, 1], "value": [25.6, 53.4, 12.7]})

with pl.SQLContext(df_a=df1, df_b=df2, eager=True) as ctx:
    result = ctx.execute("""
        SELECT
            a.id,
            a.value AS value_a,
            b.value AS value_b
        FROM df_a AS a 
        INNER JOIN df_b AS b USING (id)
        ORDER BY id
    """)

print(result)

shape: (3, 3)
┌─────┬─────────┬─────────┐
│ id  ┆ value_a ┆ value_b │
│ --- ┆ ---     ┆ ---     │
│ i64 ┆ f64     ┆ f64     │
╞═════╪═════════╪═════════╡
│ 1   ┆ 0.1     ┆ 12.7    │
│ 2   ┆ 0.2     ┆ 53.4    │
│ 3   ┆ 0.3     ┆ 25.6    │
└─────┴─────────┴─────────┘


Разбор кода:
1. Создаём фреймы данных df1, df2;
2. Создание SQL-контекста с помощью with: `with pl.SQLContext(df_a=df1, df_b=df2, eager=True) as ctx`; Здесь происходит три важные вещи:
   - df1 регистрируется в SQL-контексте под именем df_a, df2 — под именем df_b. Теперь внутри SQL-запроса можем обращаться к ним как к таблицам df_a и df_b.
   - Параметр `eager=True`. По умолчанию `.execute()` возвращает *LazyFrame*. Но здесь `eager=True`, значит результат сразу будет *DataFrame*, и вызывать `.collect()` не нужно.
   - Использование with - менеджер контекста. После завершения блока with все зарегистрированные таблицы (df_a, df_b) автоматически удаляются из памяти контекста. Это безопасно и предотвращает «засорение» при работе с множеством временных таблиц.
3. Выполнение SQL-запроса. Разберём каждую часть в порядке выполнения:
   - ` FROM df_a AS a` - Берём таблицу df_a (это df1) и даём ей псевдоним a для краткости.
   - `INNER JOIN df_b AS b USING (id)` - Присоединяем таблицу df_b (df2) по столбцу id. `USING (id)` - это сокращение от `ON a.id = b.id`. `INNER JOIN` означает: оставить только те строки, где id есть в обеих таблицах.
   - ` SELECT a.id, a.value AS value_a, b.value AS value_b` - Выбираем:
     - id из первой таблицы,
     - value из первой таблицы переименовываем в value_a,
     - value из второй таблицы переименовываем в value_b.
   - `ORDER BY id` - Сортируем результат по возрастанию id.
5. Выводим результат выполнения `print(result)`

PolarsSQL будем изучать на следующих данных: 

In [4]:
df = pl.read_csv("https://raw.githubusercontent.com/m-ardat/Library_Polars/main/dataset/diamonds.csv")
old_name = df.columns[0]
df = df.rename({old_name: "id"})

df.head(4)

id,carat,cut,color,clarity,depth,table,price,x,y,z
i64,f64,str,str,str,f64,f64,i64,f64,f64,f64
1,0.23,"""Ideal""","""E""","""SI2""",61.5,55.0,326,3.95,3.98,2.43
2,0.21,"""Premium""","""E""","""SI1""",59.8,61.0,326,3.89,3.84,2.31
3,0.23,"""Good""","""E""","""VS1""",56.9,65.0,327,4.05,4.07,2.31
4,0.29,"""Premium""","""I""","""VS2""",62.4,58.0,334,4.2,4.23,2.63


Будем работать с одним из классических наборов данных в статистике, содержащим информацию о бриллиантах. Описание можно посмотреть [здесь](https://www.kaggle.com/datasets/shivam2503/diamonds).

Краткое описание данных:
- **price**: цена в долларах США
- **carat**: вес алмаза
- **cut**: качество огранки
- **color**: цвет алмаза
- **clarity**: измерение прозрачности алмаза
- **x**: длина в мм
- **y**: ширина в мм
- **z**: глубина в мм
- **depth**: глубина
- **table**: ширина верхней части алмаза относительно самой широкой точки

### Извлечение данных

**SQL-запрос** — это команда для работы с данными: их выборки, фильтрации, агрегации и т.д.
Он состоит из операторов (ключевых слов, например SELECT, WHERE, GROUP BY), которые задают действия, и выражений (арифметических, логических и др.), определяющих условия и вычисления.

**Операторы** — это зарезервированные слова SQL, и их нельзя использовать в качестве имён таблиц, столбцов или других объектов.

Например, запрос ниже извлекает из таблицы diamonds все записи, где вес алмаза свыше 4 карат:

In [5]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT *
        FROM diamonds
        WHERE carat > 4
    """)

query

id,carat,cut,color,clarity,depth,table,price,x,y,z
i64,f64,str,str,str,f64,f64,i64,f64,f64,f64
25999,4.01,"""Premium""","""I""","""I1""",61.0,61.0,15223,10.14,10.1,6.17
26000,4.01,"""Premium""","""J""","""I1""",62.5,62.0,15223,10.02,9.94,6.24
27131,4.13,"""Fair""","""H""","""I1""",64.8,61.0,17329,10.0,9.85,6.43
27416,5.01,"""Fair""","""J""","""I1""",65.5,59.0,18018,10.74,10.54,6.98
27631,4.5,"""Fair""","""J""","""I1""",65.8,58.0,18531,10.23,10.16,6.72


Данный запрос состоит из ключевых слов:
- SELECT
- FROM
- WHERE

#### **Оператор SELECT**

Оператор **SELECT** используется для извлечения данных из таблицы. Пример минимального запроса: `SELECT имя_столбца FROM имя_таблицы`.

In [6]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT id
        FROM diamonds
    """)

query

id
i64
1
2
3
4
5
…
53936
53937
53938


Здесь оператор **SELECT** извлекает одно поле под названием `id` из таблицы `diamonds`. Имя поля указывается сразу после ключевого слова **SELECT**, а ключевое слово **FROM** указывает на таблицу, из которой извлекаются данные. Стоит отметить, что данный запрос извлекает все строки таблицы, не выполняя никакой фильтрации.

**Замечание.** Операторы нечувствительны к регистру, т.е. записи **SELECT**, **select**, **SeLeCT** равнозначны. 

Для извлечения нескольких полей таблицы после оператора **SELECT** необходимо через запятую перечислить их имена. Например:

In [7]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT
            id
            ,carat
            ,x
        FROM diamonds
    """)

query

id,carat,x
i64,f64,f64
1,0.23,3.95
2,0.21,3.89
3,0.23,4.05
4,0.29,4.2
5,0.31,4.34
…,…,…
53936,0.72,5.75
53937,0.72,5.69
53938,0.7,5.66


**Замечание.** Поля возвращаются именно в том порядке, в котором они указываются в самом запросе.

Чтобы выбрать все столбцы таблицы, вместо перечисления имён можно использовать символ `*`:

In [8]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT *
        FROM diamonds
    """)

query

id,carat,cut,color,clarity,depth,table,price,x,y,z
i64,f64,str,str,str,f64,f64,i64,f64,f64,f64
1,0.23,"""Ideal""","""E""","""SI2""",61.5,55.0,326,3.95,3.98,2.43
2,0.21,"""Premium""","""E""","""SI1""",59.8,61.0,326,3.89,3.84,2.31
3,0.23,"""Good""","""E""","""VS1""",56.9,65.0,327,4.05,4.07,2.31
4,0.29,"""Premium""","""I""","""VS2""",62.4,58.0,334,4.2,4.23,2.63
5,0.31,"""Good""","""J""","""SI2""",63.3,58.0,335,4.34,4.35,2.75
…,…,…,…,…,…,…,…,…,…,…
53936,0.72,"""Ideal""","""D""","""SI1""",60.8,57.0,2757,5.75,5.76,3.5
53937,0.72,"""Good""","""D""","""SI1""",63.1,55.0,2757,5.69,5.75,3.61
53938,0.7,"""Very Good""","""D""","""SI1""",62.8,60.0,2757,5.66,5.68,3.56


**Примечание.** Несмотря на то что символ `*` экономит время на перечисление столбцов, он редко используется в производственных запросах: выборка ненужных полей снижает производительность. Однако у него есть преимущество: он позволяет получить все поля таблицы, даже если их имена заранее неизвестны.

#### **Извлечение уникальных записей**

Ключевое слово **DISTINCT** используется для выборки только уникальных значений в указанных столбцах. Оно ставится сразу после **SELECT**:

In [9]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT DISTINCT cut
        FROM diamonds
    """)

print(query)

shape: (5, 1)
┌───────────┐
│ cut       │
│ ---       │
│ str       │
╞═══════════╡
│ Ideal     │
│ Premium   │
│ Good      │
│ Very Good │
│ Fair      │
└───────────┘


Данным запросом получены все виды огранки, которые содержатся в таблице diamonds.

**Замечение.** **DISTINCT** учитывает все указанные столбцы: строка считается дубликатом, только если все её значения совпадают с другой строкой. Пример:

In [10]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT DISTINCT cut, color
        FROM diamonds
    """)

print(query)

shape: (35, 2)
┌─────────┬───────┐
│ cut     ┆ color │
│ ---     ┆ ---   │
│ str     ┆ str   │
╞═════════╪═══════╡
│ Ideal   ┆ E     │
│ Premium ┆ E     │
│ Good    ┆ E     │
│ Premium ┆ I     │
│ Good    ┆ J     │
│ …       ┆ …     │
│ Good    ┆ G     │
│ Fair    ┆ G     │
│ Fair    ┆ J     │
│ Fair    ┆ I     │
│ Fair    ┆ D     │
└─────────┴───────┘


Здесь, например, вторая и четвертая записи считаются разными, так как их значения совпадают не по всем полям, а только по одному.

#### **Контроль размера выборки**

Для получения ограниченного числа строк (например, первых N записей) используется ключевое слово **LIMIT**:

In [11]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT color
        FROM diamonds
        LIMIT 4
    """)

print(query)

shape: (4, 1)
┌───────┐
│ color │
│ ---   │
│ str   │
╞═══════╡
│ E     │
│ E     │
│ E     │
│ I     │
└───────┘


Данный запрос извлекает поле color таблицы diamonds, а выражение `LIMIT 4` говорит о том, что должно быть извлечено не более четырех записей. Если необходимо пропустить некоторое количество строк (n строк), а затем получить следующие записи (m записей), можно задать начальное значение извлечения с помощью ключевого слова **OFFSET**.

In [12]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT color
        FROM diamonds
        LIMIT 4
        OFFSET 3
    """)

print(query)

shape: (4, 1)
┌───────┐
│ color │
│ ---   │
│ str   │
╞═══════╡
│ I     │
│ J     │
│ J     │
│ I     │
└───────┘


Выражение `LIMIT 4 OFFSET 3` говорит о том, что должно быть извлечено четыре записи, начиная с записи с индексом три. Таким образом, первое число - это количество строк для извлечения, а второе - стартовое значение.

**Замечания.** 
1. Записи при использовании ключевого слова **OFFSET** индексируются с нуля.
2. Ключевое слово **LIMIT** лишь ограничивает количество извлекаемых записей, а не определяет их количество: если в таблице содержится восемь записей, а ограничивающим значением является число сто, то извлечены будут все восемь записей и только они.

#### **Псевдонимы**

Для изменения имён столбцов в результате запроса используются псевдонимы. Они задаются с помощью ключевого слова **AS**:

In [13]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id
            ,carat AS weight
            ,price AS 'price stone'
        FROM diamonds
        LIMIT 4
    """)

print(query)

shape: (4, 3)
┌─────┬────────┬─────────────┐
│ id  ┆ weight ┆ price stone │
│ --- ┆ ---    ┆ ---         │
│ i64 ┆ f64    ┆ i64         │
╞═════╪════════╪═════════════╡
│ 1   ┆ 0.23   ┆ 326         │
│ 2   ┆ 0.21   ┆ 326         │
│ 3   ┆ 0.23   ┆ 327         │
│ 4   ┆ 0.29   ┆ 334         │
└─────┴────────┴─────────────┘


В данном запросе заменены наименование полей:
- carat на weight,
- price на price stone. Если псевдоним состоит из нескольких слов, то его необходимо обернуть, например, одинарными кавычками.

**Примечания.**
1. Записи в результирующей таблице могут отображаться в произвольном порядке.
2. Все лишние пробельные символы в запросе пропускаются при его обработке, поэтому запрос может быть записан как в одной длинной строке, так и разбит на несколько строк.
3. PolarsSQL поддерживает несколько видов комментариев:
   - Однострочный комментарий: `-- тело однострочного комментария`;
   - Многострочный комментарий: `/* тело многострочного комментария */`.
4. Оператор **SELECT** можно использовать для вывода конкретных значений, например, строк или чисел.

In [14]:
with pl.SQLContext(t=df, eager=True) as ctx:
    query = ctx.execute("""
        -- Выведем константу
        SELECT 1 AS const
        /* Данный запрос
        создаёт таблицу с полем const, 
        в котором записано значение 1
        */
    """)

query

const
i32
1


### Сортировка данных

#### **Оператор ORDER BY**

Зачастую данные необходимо получать в определённом порядке - например, по алфавиту или по возрастанию возраста. В PolarsSQL для этого используется оператор **ORDER BY**. Достаточно добавить его в конец запроса и указать столбец, по которому требуется сортировка:

In [15]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, carat, price
        FROM diamonds
        WHERE carat > 4
        ORDER BY carat
    """)

print(query)

shape: (5, 3)
┌───────┬───────┬───────┐
│ id    ┆ carat ┆ price │
│ ---   ┆ ---   ┆ ---   │
│ i64   ┆ f64   ┆ i64   │
╞═══════╪═══════╪═══════╡
│ 25999 ┆ 4.01  ┆ 15223 │
│ 26000 ┆ 4.01  ┆ 15223 │
│ 27131 ┆ 4.13  ┆ 17329 │
│ 27631 ┆ 4.5   ┆ 18531 │
│ 27416 ┆ 5.01  ┆ 18018 │
└───────┴───────┴───────┘


В данном примере извлекаются данные о камнях, вес которых свыше четырёх карат. Результирующая выборка за счет **ORDER BY** отсортирована в порядке возрастания значения поля carat. По умолчанию оператор ORDER BY выполняет сортировку именно по возрастанию. Если хотим в обратном порядке (по убыванию), то необходимо после имени поля указать ключевое слово **DESC**.

In [16]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, carat, price
        FROM diamonds
        WHERE carat > 4
        ORDER BY carat DESC
    """)

print(query)

shape: (5, 3)
┌───────┬───────┬───────┐
│ id    ┆ carat ┆ price │
│ ---   ┆ ---   ┆ ---   │
│ i64   ┆ f64   ┆ i64   │
╞═══════╪═══════╪═══════╡
│ 27416 ┆ 5.01  ┆ 18018 │
│ 27631 ┆ 4.5   ┆ 18531 │
│ 27131 ┆ 4.13  ┆ 17329 │
│ 25999 ┆ 4.01  ┆ 15223 │
│ 26000 ┆ 4.01  ┆ 15223 │
└───────┴───────┴───────┘


**Примечания.** 
1. В запросе оператор **ORDER BY** должен следовать после операторов **SELECT**, **FROM** и **WHERE**, иначе запрос завершится ошибкой.
2. Поля, по которым выполняется сортировка, необязательно должны быть извлечены, то есть сортировка может выполняться и по тем полям, которые не попадают в результирующую таблицу.

In [17]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, carat, price
        FROM diamonds
        WHERE carat > 4
        ORDER BY color DESC
    """)

print(query)

shape: (5, 3)
┌───────┬───────┬───────┐
│ id    ┆ carat ┆ price │
│ ---   ┆ ---   ┆ ---   │
│ i64   ┆ f64   ┆ i64   │
╞═══════╪═══════╪═══════╡
│ 26000 ┆ 4.01  ┆ 15223 │
│ 27416 ┆ 5.01  ┆ 18018 │
│ 27631 ┆ 4.5   ┆ 18531 │
│ 25999 ┆ 4.01  ┆ 15223 │
│ 27131 ┆ 4.13  ┆ 17329 │
└───────┴───────┴───────┘


Когда сортировки по одному полю недостаточно (например, из-за одинаковых значений), можно сортировать по нескольким столбцам. Для этого их перечисляют через запятую после **ORDER BY**:

In [18]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            color, carat, price
        FROM diamonds
        WHERE carat > 4
        ORDER BY color DESC, price
    """)

print(query)

shape: (5, 3)
┌───────┬───────┬───────┐
│ color ┆ carat ┆ price │
│ ---   ┆ ---   ┆ ---   │
│ str   ┆ f64   ┆ i64   │
╞═══════╪═══════╪═══════╡
│ J     ┆ 4.01  ┆ 15223 │
│ J     ┆ 5.01  ┆ 18018 │
│ J     ┆ 4.5   ┆ 18531 │
│ I     ┆ 4.01  ┆ 15223 │
│ H     ┆ 4.13  ┆ 17329 │
└───────┴───────┴───────┘


Запрос сортирует сначала по color, а при совпадении значений - по price: записи с разными цветами упорядочиваются по цвету, а записи с одинаковым цветом - по цене.

**Замечание.** Ключевое слово **DESC** применяется только к тому полю, после которого стоит.

Операторы **ORDER BY** и **LIMIT** можно комбинировать. Например, чтобы найти самый тяжёлый камень:
- сначала сортируем по весу в порядке убывания,
- затем ограничиваем результат одной строкой.

In [19]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            carat, price
        FROM diamonds
        WHERE carat > 4
        ORDER BY carat DESC
        LIMIT 1
    """)

print(query)

shape: (1, 2)
┌───────┬───────┐
│ carat ┆ price │
│ ---   ┆ ---   │
│ f64   ┆ i64   │
╞═══════╪═══════╡
│ 5.01  ┆ 18018 │
└───────┴───────┘


Так как сортировка является одной из завершающих операций и выполняется уже после формирования результирующей таблицы, то в **ORDER BY** можно использовать как наименование полей, так и их псевдонимы.

In [20]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            carat AS stone_weight, color AS c, price
        FROM diamonds
        WHERE carat > 4
        ORDER BY stone_weight, color
        LIMIT 3
    """)

print(query)

shape: (3, 3)
┌──────────────┬─────┬───────┐
│ stone_weight ┆ c   ┆ price │
│ ---          ┆ --- ┆ ---   │
│ f64          ┆ str ┆ i64   │
╞══════════════╪═════╪═══════╡
│ 4.01         ┆ I   ┆ 15223 │
│ 4.01         ┆ J   ┆ 15223 │
│ 4.13         ┆ H   ┆ 17329 │
└──────────────┴─────┴───────┘


**Примечание.** В PolarsSQL при сортировке NULL значений (пропусков в данных) ведёт себя особым образом:
- при сортировки по возрастанию пропуски оказываются в конце;
- при сортировки по убыванию пропуски оказываются в начале.

In [21]:
data_null = {
    "id": [1, 2, 3, 4, 5],
    "name": ["Alice", "Bob", None, "David", "Eve"],
    "value": [10, None, 30, 20, None]
}

dfnull = pl.DataFrame(data_null)

with pl.SQLContext(t=dfnull, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, value
        FROM t
        ORDER BY value
    """)

print(query)

shape: (5, 2)
┌─────┬───────┐
│ id  ┆ value │
│ --- ┆ ---   │
│ i64 ┆ i64   │
╞═════╪═══════╡
│ 1   ┆ 10    │
│ 4   ┆ 20    │
│ 3   ┆ 30    │
│ 2   ┆ null  │
│ 5   ┆ null  │
└─────┴───────┘


In [22]:
with pl.SQLContext(t=dfnull, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, value
        FROM t
        ORDER BY value DESC
    """)

print(query)

shape: (5, 2)
┌─────┬───────┐
│ id  ┆ value │
│ --- ┆ ---   │
│ i64 ┆ i64   │
╞═════╪═══════╡
│ 2   ┆ null  │
│ 5   ┆ null  │
│ 3   ┆ 30    │
│ 4   ┆ 20    │
│ 1   ┆ 10    │
└─────┴───────┘


При сортировке с помощью **ORDER BY** можно явно указать положение строк с NULL. Ключевая фраза **NULLS LAST** перемещает все значения NULL в конец результата, а **NULLS FIRST** - в начало.

Пример:

In [106]:
with pl.SQLContext(t=dfnull, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, value
        FROM t
        ORDER BY value DESC NULLS LAST
    """)

print(query)

shape: (5, 2)
┌─────┬───────┐
│ id  ┆ value │
│ --- ┆ ---   │
│ i64 ┆ i64   │
╞═════╪═══════╡
│ 3   ┆ 30    │
│ 4   ┆ 20    │
│ 1   ┆ 10    │
│ 2   ┆ null  │
│ 5   ┆ null  │
└─────┴───────┘


### Фильтрация данных

**Оператор WHERE**

В данных редко требуется выбирать все записи таблицы. Чаще нужны только те, что соответствуют определённым условиям. Оператор **WHERE** фильтрует строки: в результат попадают только те записи, которые удовлетворяют заданному условию, остальные отбрасываются. 

In [23]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, carat, price
        FROM diamonds
        WHERE carat = 4
        ORDER BY carat
    """)

print(query)

shape: (1, 3)
┌───────┬───────┬───────┐
│ id    ┆ carat ┆ price │
│ ---   ┆ ---   ┆ ---   │
│ i64   ┆ f64   ┆ i64   │
╞═══════╪═══════╪═══════╡
│ 26445 ┆ 4.0   ┆ 15984 │
└───────┴───────┴───────┘


В примере выше извлекли данные о бриллиантах, вес которых ровно 4 карата, т.е. провели простую проверку на равенство веса четырём.

**Замечание.** Оператор **WHERE** указывается после **FROM** и перед **ORDER BY**.

#### Операторы сравнения

PolarsSQL поддерживает целый ряд операторов сравнения после оператора **WHERE**:
- `=` -	равно;
- `<>` или `!=` - неравно;
- `<=>` - эквивалентно;
- `<` - меньше;
- `>` - больше;
- `<=` - меньше или равно;
- `>` - больше;
- `>=` - больше или равно;
- `BETWEEN` - входит в промежуток;
- `IS NULL` - значение есть NULL;
- `IS NOT NULL` - значение не является NULL;

Рассмотрим пару примеров.

Пример 1. Извлечем из таблицы diamonds данные о камнях, у которых оганка (cut) не является "Ideal":

In [24]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, carat, price
        FROM diamonds
        WHERE cut <> 'Ideal'
        ORDER BY carat
    """)

print(query)

shape: (32_389, 3)
┌───────┬───────┬───────┐
│ id    ┆ carat ┆ price │
│ ---   ┆ ---   ┆ ---   │
│ i64   ┆ f64   ┆ i64   │
╞═══════╪═══════╪═══════╡
│ 15    ┆ 0.2   ┆ 345   │
│ 31592 ┆ 0.2   ┆ 367   │
│ 31593 ┆ 0.2   ┆ 367   │
│ 31594 ┆ 0.2   ┆ 367   │
│ 31595 ┆ 0.2   ┆ 367   │
│ …     ┆ …     ┆ …     │
│ 25999 ┆ 4.01  ┆ 15223 │
│ 26000 ┆ 4.01  ┆ 15223 │
│ 27131 ┆ 4.13  ┆ 17329 │
│ 27631 ┆ 4.5   ┆ 18531 │
│ 27416 ┆ 5.01  ┆ 18018 │
└───────┴───────┴───────┘


Пример 2. Найдём все камни, вес которых от 2 до 3 карат включительно. В этом нам поможет оператор **BETWEEN**.

In [25]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, carat, price
        FROM diamonds
        WHERE carat BETWEEN 2 AND 3
        ORDER BY carat
    """)

print(query)

shape: (2_122, 3)
┌───────┬───────┬───────┐
│ id    ┆ carat ┆ price │
│ ---   ┆ ---   ┆ ---   │
│ i64   ┆ f64   ┆ i64   │
╞═══════╪═══════╪═══════╡
│ 11635 ┆ 2.0   ┆ 5051  │
│ 13930 ┆ 2.0   ┆ 5667  │
│ 14646 ┆ 2.0   ┆ 5914  │
│ 15862 ┆ 2.0   ┆ 6344  │
│ 16299 ┆ 2.0   ┆ 6521  │
│ …     ┆ …     ┆ …     │
│ 23540 ┆ 3.0   ┆ 11548 │
│ 24817 ┆ 3.0   ┆ 13203 │
│ 25851 ┆ 3.0   ┆ 14918 │
│ 26933 ┆ 3.0   ┆ 16970 │
│ 26934 ┆ 3.0   ┆ 16970 │
└───────┴───────┴───────┘


Синтаксис немного отличается от других операторов, так как **BETWEEN** требует два значения: начальное и конечное, которые в запросе разделены словом **AND**. Оператор **BETWEEN** включает в промежуток граничные значения.

Чтобы проверить отсутсвуют ли значения в данных есть специальный оператор **IS NULL**, и противоположный ему - **IS NOT NULL**, который определяет не пустые строки.

In [26]:
data_null = {
    "id": [1, 2, 3, 4, 5],
    "name": ["Alice", "Bob", None, "David", "Eve"],
    "value": [10, None, 30, 20, None]
}

dfnull = pl.DataFrame(data_null)

with pl.SQLContext(t=dfnull, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, value
        FROM t
        WHERE value IS NULL
    """)

print(query)

shape: (2, 2)
┌─────┬───────┐
│ id  ┆ value │
│ --- ┆ ---   │
│ i64 ┆ i64   │
╞═════╪═══════╡
│ 2   ┆ null  │
│ 5   ┆ null  │
└─────┴───────┘


Вывели строки, где value содержит пропуски в данных.

#### Логические операторы

PolarsSQL есть следующие логические операторы:
- `AND` - логическое И;
- `OR` - логическое ИЛИ;
- `NOT` - Логическое НЕ;
- `IN` - Вхождение.

Фильтрация данных при их извлечении не ограничивается лишь одним критерием. Чтобы использовать более сложные фильтры есть логические операторы **AND** и **OR**.

Оператор **AND** используется для извлечения записей, которые удовлетворяют всем заданным условиям.

In [27]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, carat, price, color
        FROM diamonds
        WHERE 1=1
            AND carat >= 4
            AND color = 'J'
        ORDER BY price DESC
    """)

print(query)

shape: (3, 4)
┌───────┬───────┬───────┬───────┐
│ id    ┆ carat ┆ price ┆ color │
│ ---   ┆ ---   ┆ ---   ┆ ---   │
│ i64   ┆ f64   ┆ i64   ┆ str   │
╞═══════╪═══════╪═══════╪═══════╡
│ 27631 ┆ 4.5   ┆ 18531 ┆ J     │
│ 27416 ┆ 5.01  ┆ 18018 ┆ J     │
│ 26000 ┆ 4.01  ┆ 15223 ┆ J     │
└───────┴───────┴───────┴───────┘


В данном запросе извлекаем все записи, где вес камня не менее 4 карат и  цвет J. В операторе **WHERE** содержатся два условия, которые соединены через **AND**: необходимо извлечь записи, которые удовлетворяют всем заданным условиям. Запись 1=1 писать не обязательно, так как это условие всегда верно, и используется только лишь для удобства.

Оператор **OR** работает иначе: извлекаем строки, которые удовлетворяют хотя бы одному условию из заданных.

In [28]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, carat, price, color
        FROM diamonds
        WHERE carat >= 4 OR price >= 18796
        ORDER BY price DESC
    """)

print(query)

shape: (12, 4)
┌───────┬───────┬───────┬───────┐
│ id    ┆ carat ┆ price ┆ color │
│ ---   ┆ ---   ┆ ---   ┆ ---   │
│ i64   ┆ f64   ┆ i64   ┆ str   │
╞═══════╪═══════╪═══════╪═══════╡
│ 27750 ┆ 2.29  ┆ 18823 ┆ I     │
│ 27749 ┆ 2.0   ┆ 18818 ┆ G     │
│ 27748 ┆ 1.51  ┆ 18806 ┆ G     │
│ 27747 ┆ 2.07  ┆ 18804 ┆ G     │
│ 27746 ┆ 2.0   ┆ 18803 ┆ H     │
│ …     ┆ …     ┆ …     ┆ …     │
│ 27416 ┆ 5.01  ┆ 18018 ┆ J     │
│ 27131 ┆ 4.13  ┆ 17329 ┆ H     │
│ 26445 ┆ 4.0   ┆ 15984 ┆ I     │
│ 25999 ┆ 4.01  ┆ 15223 ┆ I     │
│ 26000 ┆ 4.01  ┆ 15223 ┆ J     │
└───────┴───────┴───────┴───────┘


**Примечание.**
**WHERE** может содержать любое количество логических операторв **AND** и **OR**, при помощи которых можно создать сложные фильтры. Стоит помнить, что сперва обрабатывается **AND**, а затем **OR**.

Оператор **IN** позволяет определить, входит ли значение поля с одним из перечисленных значений.

In [29]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, carat, price, color
        FROM diamonds
        WHERE color IN ('J', 'H', 'D')
        ORDER BY price DESC
        LIMIT 5
    """)

print(query)

shape: (5, 4)
┌───────┬───────┬───────┬───────┐
│ id    ┆ carat ┆ price ┆ color │
│ ---   ┆ ---   ┆ ---   ┆ ---   │
│ i64   ┆ f64   ┆ i64   ┆ str   │
╞═══════╪═══════╪═══════╪═══════╡
│ 27746 ┆ 2.0   ┆ 18803 ┆ H     │
│ 27743 ┆ 2.04  ┆ 18795 ┆ H     │
│ 27737 ┆ 2.03  ┆ 18781 ┆ H     │
│ 27731 ┆ 2.08  ┆ 18760 ┆ H     │
│ 27727 ┆ 2.36  ┆ 18745 ┆ H     │
└───────┴───────┴───────┴───────┘


Данный запрос извлекает данные о пяти самых дорогих камнях, которых имеют цвет J, H, D.

Логический оператор **NOT** служит для одной цели - отрицать заданное условие. Например, с помощью данного оператора можно извлечь данные о пяти самых дорогих комнях, цвет которых не которых имеют цвет J, H, D.

In [30]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, carat, price, color
        FROM diamonds
        WHERE color NOT IN ('J', 'H', 'D')
        ORDER BY price DESC
        LIMIT 5
    """)

print(query)

shape: (5, 4)
┌───────┬───────┬───────┬───────┐
│ id    ┆ carat ┆ price ┆ color │
│ ---   ┆ ---   ┆ ---   ┆ ---   │
│ i64   ┆ f64   ┆ i64   ┆ str   │
╞═══════╪═══════╪═══════╪═══════╡
│ 27750 ┆ 2.29  ┆ 18823 ┆ I     │
│ 27749 ┆ 2.0   ┆ 18818 ┆ G     │
│ 27748 ┆ 1.51  ┆ 18806 ┆ G     │
│ 27747 ┆ 2.07  ┆ 18804 ┆ G     │
│ 27745 ┆ 2.29  ┆ 18797 ┆ I     │
└───────┴───────┴───────┴───────┘


#### Метасимволы и оператор LIKE

Оператор **LIKE** используется для поиска по шаблону в строковых данных, когда точное значение неизвестно. Он применяется вместе с метасимволами - специальными знаками, которые заменяют другие символы:
- `%` - заменяет любую последовательность символов (включая пустую);
- `_`- соответствие одному любому символу.

Шаблоны с **LIKE** работают только со строковыми полями и позволяют гибко фильтровать данные по частичному совпадению.

In [31]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, carat, price, cut
        FROM diamonds
        WHERE cut LIKE 'P%'
        ORDER BY price DESC
        LIMIT 5
    """)

print(query)

shape: (5, 4)
┌───────┬───────┬───────┬─────────┐
│ id    ┆ carat ┆ price ┆ cut     │
│ ---   ┆ ---   ┆ ---   ┆ ---     │
│ i64   ┆ f64   ┆ i64   ┆ str     │
╞═══════╪═══════╪═══════╪═════════╡
│ 27750 ┆ 2.29  ┆ 18823 ┆ Premium │
│ 27745 ┆ 2.29  ┆ 18797 ┆ Premium │
│ 27743 ┆ 2.04  ┆ 18795 ┆ Premium │
│ 27744 ┆ 2.0   ┆ 18795 ┆ Premium │
│ 27741 ┆ 1.71  ┆ 18791 ┆ Premium │
└───────┴───────┴───────┴─────────┘


`WHERE cut LIKE 'P%'` найдёт все качества огранки, которые начинаются с «P».

In [32]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT
            id, carat, price, cut
        FROM diamonds
        WHERE cut LIKE '____'
        ORDER BY price DESC
        LIMIT 5
    """)

print(query)

shape: (5, 4)
┌───────┬───────┬───────┬──────┐
│ id    ┆ carat ┆ price ┆ cut  │
│ ---   ┆ ---   ┆ ---   ┆ ---  │
│ i64   ┆ f64   ┆ i64   ┆ str  │
╞═══════╪═══════╪═══════╪══════╡
│ 27740 ┆ 2.8   ┆ 18788 ┆ Good │
│ 27683 ┆ 2.07  ┆ 18707 ┆ Good │
│ 27673 ┆ 2.67  ┆ 18686 ┆ Good │
│ 27661 ┆ 2.01  ┆ 18640 ┆ Good │
│ 27659 ┆ 2.01  ┆ 18625 ┆ Good │
└───────┴───────┴───────┴──────┘


`WHERE cut LIKE '____'` найдёт все качества огранки, у которых наименование ровно четыре символа (любых).

#### **Примечания.**

1. Псевдонимы полей в PolarsSQL не могут быть использованы в блоке **WHERE** для фильтрации.
2. PolarsSQL учитывают регистр при сравнении строковых значений.
3. Порядок операторов в SQL-запросе: **SELECT** → **FROM** → **WHERE** → **ORDER BY** → **LIMIT**.
4. Логические операторы **AND**, **OR**, **NOT**, **IN** и **NOT IN** имеют разный приоритет. В порядке уменьшения их приоритета:
    - IN , NOT IN
    - NOT
    - AND
    - OR
5. Логический оператор **NOT** отрицает только то условие, перед которым указан.
6. Метасимволы могут встречаться в любом месте шаблона поиска, причем в неограниченном количестве.
7. Важная особенность оператора **LIKE**: при поиске с его помощью строк, соответствующих шаблону, регистр в PolarsSQL учитывается.
8. Для поиска самого метасивола необходимо использовать экранирование.


### Создание новых признаков

**Вычисляемые поля и функции**

**Вычисляемые поля** - поля, значения которых не хранятся в таблице, а рассчитываются на основе других столбцов с помощью выражений: математических операций, строковых преобразований или функций.

Например:
- Объединить имя и фамилию в одно поле.
- Посчитать общую стоимость как цена помноженное на количество.
  
**Функции** - готовые операции, которые принимают аргументы и возвращают результат.

Вычисляемые поля позволяют получать нужную информацию «на лету», не изменяя исходные данные.

#### Функция CONCAT() 
В PolarsSQL функция **CONCAT()** объединяет (конкатенирует) значения нескольких полей в одну строку. Аргументы могут быть любого типа - они автоматически преобразуются в строки.

Пример:

In [33]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT
            id, cut, color, price
            ,CONCAT(id, cut, color, price) AS gluing
        FROM diamonds
        LIMIT 5
    """)

print(query)

shape: (5, 5)
┌─────┬─────────┬───────┬───────┬──────────────┐
│ id  ┆ cut     ┆ color ┆ price ┆ gluing       │
│ --- ┆ ---     ┆ ---   ┆ ---   ┆ ---          │
│ i64 ┆ str     ┆ str   ┆ i64   ┆ str          │
╞═════╪═════════╪═══════╪═══════╪══════════════╡
│ 1   ┆ Ideal   ┆ E     ┆ 326   ┆ 1IdealE326   │
│ 2   ┆ Premium ┆ E     ┆ 326   ┆ 2PremiumE326 │
│ 3   ┆ Good    ┆ E     ┆ 327   ┆ 3GoodE327    │
│ 4   ┆ Premium ┆ I     ┆ 334   ┆ 4PremiumI334 │
│ 5   ┆ Good    ┆ J     ┆ 335   ┆ 5GoodJ335    │
└─────┴─────────┴───────┴───────┴──────────────┘


Аналогичную операцию можно выполнить при помощи оператора `||`:

In [34]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT
            id, cut, color, price
            ,id||cut||color||price AS gluing
        FROM diamonds
        LIMIT 5
    """)

print(query)

shape: (5, 5)
┌─────┬─────────┬───────┬───────┬──────────────┐
│ id  ┆ cut     ┆ color ┆ price ┆ gluing       │
│ --- ┆ ---     ┆ ---   ┆ ---   ┆ ---          │
│ i64 ┆ str     ┆ str   ┆ i64   ┆ str          │
╞═════╪═════════╪═══════╪═══════╪══════════════╡
│ 1   ┆ Ideal   ┆ E     ┆ 326   ┆ 1IdealE326   │
│ 2   ┆ Premium ┆ E     ┆ 326   ┆ 2PremiumE326 │
│ 3   ┆ Good    ┆ E     ┆ 327   ┆ 3GoodE327    │
│ 4   ┆ Premium ┆ I     ┆ 334   ┆ 4PremiumI334 │
│ 5   ┆ Good    ┆ J     ┆ 335   ┆ 5GoodJ335    │
└─────┴─────────┴───────┴───────┴──────────────┘


Ограничений на количество используемых вычисляемых полей нет. Запрос может включать любое количество.

#### Функция CONCAT_WS()

**CONCAT_WS()** объединяет (конкатенирует) несколько строк, вставляя между ними указанный разделитель.

Синтаксис: `CONCAT_WS(separator, str1, str2, ..., strN)`
- Первый аргумент - разделитель.
- Последующие аргументы - строки для объединения.
- Игнорирует NULL-значения (не вставляет разделитель вместо них).
- Если все строки NULL, результат - пустая строка.
  
Пример:

In [35]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            cut, color, price
            ,CONCAT(cut, color, price) AS bas_concat
            ,CONCAT_WS(', ', cut, color, price) AS concat_ws
            ,CONCAT_WS(', ', NULL, color, price) AS null_color_price
            ,CONCAT_WS(', ', NULL, NULL) AS null_null
        FROM diamonds
        LIMIT 5
    """)

print(query)

shape: (5, 7)
┌─────────┬───────┬───────┬─────────────┬─────────────────┬──────────────────┬───────────┐
│ cut     ┆ color ┆ price ┆ bas_concat  ┆ concat_ws       ┆ null_color_price ┆ null_null │
│ ---     ┆ ---   ┆ ---   ┆ ---         ┆ ---             ┆ ---              ┆ ---       │
│ str     ┆ str   ┆ i64   ┆ str         ┆ str             ┆ str              ┆ str       │
╞═════════╪═══════╪═══════╪═════════════╪═════════════════╪══════════════════╪═══════════╡
│ Ideal   ┆ E     ┆ 326   ┆ IdealE326   ┆ Ideal, E, 326   ┆ E, 326           ┆           │
│ Premium ┆ E     ┆ 326   ┆ PremiumE326 ┆ Premium, E, 326 ┆ E, 326           ┆           │
│ Good    ┆ E     ┆ 327   ┆ GoodE327    ┆ Good, E, 327    ┆ E, 327           ┆           │
│ Premium ┆ I     ┆ 334   ┆ PremiumI334 ┆ Premium, I, 334 ┆ I, 334           ┆           │
│ Good    ┆ J     ┆ 335   ┆ GoodJ335    ┆ Good, J, 335    ┆ J, 335           ┆           │
└─────────┴───────┴───────┴─────────────┴─────────────────┴─────────────────

Функция **CONCAT_WS()**, как и **CONCAT()**, умеет работать с аргументами любых типов, поскольку перед объединением неявно преобразует все аргументы в строки.

#### Математические операции

С помощью вычисляемых полей можно выполнять математические операции над данными. Например, найдём объём камней:

In [36]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            id, carat, price, x, y, z,
            x*y*z AS volume
        FROM diamonds
        LIMIT 5
    """)

print(query)

shape: (5, 7)
┌─────┬───────┬───────┬──────┬──────┬──────┬───────────┐
│ id  ┆ carat ┆ price ┆ x    ┆ y    ┆ z    ┆ volume    │
│ --- ┆ ---   ┆ ---   ┆ ---  ┆ ---  ┆ ---  ┆ ---       │
│ i64 ┆ f64   ┆ i64   ┆ f64  ┆ f64  ┆ f64  ┆ f64       │
╞═════╪═══════╪═══════╪══════╪══════╪══════╪═══════════╡
│ 1   ┆ 0.23  ┆ 326   ┆ 3.95 ┆ 3.98 ┆ 2.43 ┆ 38.20203  │
│ 2   ┆ 0.21  ┆ 326   ┆ 3.89 ┆ 3.84 ┆ 2.31 ┆ 34.505856 │
│ 3   ┆ 0.23  ┆ 327   ┆ 4.05 ┆ 4.07 ┆ 2.31 ┆ 38.076885 │
│ 4   ┆ 0.29  ┆ 334   ┆ 4.2  ┆ 4.23 ┆ 2.63 ┆ 46.72458  │
│ 5   ┆ 0.31  ┆ 335   ┆ 4.34 ┆ 4.35 ┆ 2.75 ┆ 51.91725  │
└─────┴───────┴───────┴──────┴──────┴──────┴───────────┘


В PolarsSQL поддерживаются основные математические операторы: 
- `+` (сложение),
- `-` (вычитание),
- `*` (умножение)
- `/`(деление).

#### Примечания.

1. Если в математическом выражении есть NULL, результат всегда NULL.
2. Деление на ноль не вызывает ошибку, а возвращает `inf` (бесконечность).
3. Псевдонимы вычисляемых полей не могут быть использованы в блоке оператора **WHERE** для фильтрации записей.
4. Поведение функции **CONCAT()** при наличии в аргументах NULL аналогично **CONCAT_WS()**.

In [37]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            5 * NULL AS multiplication,
            4 - NULL AS subtraction,
            3 + NULL AS addition,
            NULL / 2 AS division,
            carat / 0 AS division_by_zero,
            CONCAT(color, NULL) AS color_null,
            CONCAT(NULL, NULL) AS null_null
        FROM diamonds
        LIMIT 1
    """)

print(query)

shape: (1, 7)
┌────────────────┬─────────────┬──────────┬──────────┬──────────────────┬────────────┬───────────┐
│ multiplication ┆ subtraction ┆ addition ┆ division ┆ division_by_zero ┆ color_null ┆ null_null │
│ ---            ┆ ---         ┆ ---      ┆ ---      ┆ ---              ┆ ---        ┆ ---       │
│ i32            ┆ i32         ┆ i32      ┆ i32      ┆ f64              ┆ str        ┆ str       │
╞════════════════╪═════════════╪══════════╪══════════╪══════════════════╪════════════╪═══════════╡
│ null           ┆ null        ┆ null     ┆ null     ┆ inf              ┆ E          ┆           │
└────────────────┴─────────────┴──────────┴──────────┴──────────────────┴────────────┴───────────┘


### Применение встроенных функций

Как и большинство языков программирования, PolarsSQL предоставляет множество встроенных функций для работы с данными. Их можно разделить на три основные группы:
- Строковые: для обработки строк (например, удаление пробелов или приведение к верхнему регистру);
- Математические: для математических операций (возведение в степень, извлечение корня и т.п.);
- Datetime: для работы с датами.

#### Строковые функции

Рассмотрим следующие функции:
- CHAR_LENGTH()
- LOWER()
- UPPER()
- LTRIM()
- RTRIM()
- REVERSE()
- LEFT()
- RIGHT()
- REPLACE()
- SUBSTR()
- TRIM()

In [38]:
# Создаём пустой DataFrame
df_t = pl.DataFrame([])

##### Функция CHAR_LENGTH()

Функция **CHAR_LENGTH()** возвращает количество символов в переданной строке.

Пример:

In [39]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            CHAR_LENGTH('polars') AS len_polars,
            CHAR_LENGTH('sql') AS len_sql,
            CHAR_LENGTH('best') AS len_best
    """)

print(query)

shape: (1, 3)
┌────────────┬─────────┬──────────┐
│ len_polars ┆ len_sql ┆ len_best │
│ ---        ┆ ---     ┆ ---      │
│ u32        ┆ u32     ┆ u32      │
╞════════════╪═════════╪══════════╡
│ 6          ┆ 3       ┆ 4        │
└────────────┴─────────┴──────────┘


##### Функция LOWER()

Функция **LOWER()** преобразует все символы строки в нижний регистр и возвращает результат.

Пример:

In [40]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            LOWER('Polars') AS lower_polars,
            LOWER('SQL') AS lower_sql,
            LOWER('bEsT') AS lower_best
    """)

print(query)

shape: (1, 3)
┌──────────────┬───────────┬────────────┐
│ lower_polars ┆ lower_sql ┆ lower_best │
│ ---          ┆ ---       ┆ ---        │
│ str          ┆ str       ┆ str        │
╞══════════════╪═══════════╪════════════╡
│ polars       ┆ sql       ┆ best       │
└──────────────┴───────────┴────────────┘


##### Функция UPPER()

Функция **UPPER()** преобразует все символы строки в верхний регистр и возвращает результат.

Пример:

In [41]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            UPPER('Polars') AS upper_polars,
            UPPER('sql') AS upper_sql,
            UPPER('bEsT') AS upper_best
    """)

print(query)

shape: (1, 3)
┌──────────────┬───────────┬────────────┐
│ upper_polars ┆ upper_sql ┆ upper_best │
│ ---          ┆ ---       ┆ ---        │
│ str          ┆ str       ┆ str        │
╞══════════════╪═══════════╪════════════╡
│ POLARS       ┆ SQL       ┆ BEST       │
└──────────────┴───────────┴────────────┘


##### Функции LTRIM() и RTRIM()

Функции **LTRIM()** и **RTRIM()** удаляют пробелы в начале и в конце строки соответственно. **LTRIM()** — слева, **RTRIM()** — справа.

Пример:

In [42]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            -- Исходная длина
            CHAR_LENGTH('    Polars') AS len_polars,
            CHAR_LENGTH('sql           ') AS len_sql,
            -- Удаляем пробелы
            LTRIM('    Polars') AS ltrim_polars,
            RTRIM('sql           ') AS rtrim_sql,
            -- Проверим длину
            CHAR_LENGTH(LTRIM('    Polars')) AS len_ltrim_polars,
            CHAR_LENGTH(RTRIM('sql           ')) AS len_rtrim_sql
    """)

print(query)

shape: (1, 6)
┌────────────┬─────────┬──────────────┬───────────┬──────────────────┬───────────────┐
│ len_polars ┆ len_sql ┆ ltrim_polars ┆ rtrim_sql ┆ len_ltrim_polars ┆ len_rtrim_sql │
│ ---        ┆ ---     ┆ ---          ┆ ---       ┆ ---              ┆ ---           │
│ u32        ┆ u32     ┆ str          ┆ str       ┆ u32              ┆ u32           │
╞════════════╪═════════╪══════════════╪═══════════╪══════════════════╪═══════════════╡
│ 10         ┆ 14      ┆ Polars       ┆ sql       ┆ 6                ┆ 3             │
└────────────┴─────────┴──────────────┴───────────┴──────────────────┴───────────────┘


##### Функция REVERSE()

Функция **REVERSE()** возвращает строку с символами в обратном порядке.

Пример:

In [43]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            REVERSE('Polars') AS reverse_polars,
            REVERSE('SQL') AS reverse_sql,
            REVERSE('best') AS reverse_best
    """)

print(query)

shape: (1, 3)
┌────────────────┬─────────────┬──────────────┐
│ reverse_polars ┆ reverse_sql ┆ reverse_best │
│ ---            ┆ ---         ┆ ---          │
│ str            ┆ str         ┆ str          │
╞════════════════╪═════════════╪══════════════╡
│ sraloP         ┆ LQS         ┆ tseb         │
└────────────────┴─────────────┴──────────────┘


##### Функции LEFT() и RIGHT()

Функция **LEFT()** возвращает указанное количество символов с начала строки, **RIGHT()** - с конца.  Принимают два параметра в следующем порядке:
- str - исходная строка;
- count - количество извлекаемых символов.
  
Пример:

In [44]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            LEFT('Polars', 2) AS left_polars,
            RIGHT('SQL_best', 2) AS right_SQLbest
    """)

print(query)

shape: (1, 2)
┌─────────────┬───────────────┐
│ left_polars ┆ right_SQLbest │
│ ---         ┆ ---           │
│ str         ┆ str           │
╞═════════════╪═══════════════╡
│ Po          ┆ st            │
└─────────────┴───────────────┘


Если количество извлекаемых символов равно 0, функции **LEFT()** и **RIGHT()** вернут пустую строку.

In [45]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            LEFT('Polars', 0) AS left_polars,
            RIGHT('SQL_best', 0) AS right_SQLbest
    """)

print(query)

shape: (1, 2)
┌─────────────┬───────────────┐
│ left_polars ┆ right_SQLbest │
│ ---         ┆ ---           │
│ str         ┆ str           │
╞═════════════╪═══════════════╡
│             ┆               │
└─────────────┴───────────────┘


Если передать отрицательное значение, поведение может быть не таким, как ожидалось.
Например, передавая -3, **LEFT** возвращает строку, отрезанную с правой стороны до оставшихся символов: "Pol".
А функция **RIGHT** при передачи -4 вернёт строку, оставшуюся после отрезания 4 символов справа: "best".

In [46]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            LEFT('Polars', -3) AS left_polars,
            RIGHT('SQL_best', -4) AS right_SQLbest
    """)

print(query)

shape: (1, 2)
┌─────────────┬───────────────┐
│ left_polars ┆ right_SQLbest │
│ ---         ┆ ---           │
│ str         ┆ str           │
╞═════════════╪═══════════════╡
│ Pol         ┆ best          │
└─────────────┴───────────────┘


Таким образом, при извлечении символов из строк на основе отрицательных значений может привести к неожиданным результатам.

##### Функция REPLACE()

Функция **REPLACE()** заменяет все вхождения одной подстроки на другую в заданной строке и возвращает результат. Включает в себя следующие три параметра:
- str - исходная строка,
- from_str - заменяемая подстрока,
- to_str - заменяющая подстрока.

Пример:

In [47]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            REPLACE('Polars', 'rs', '') AS replace_polars,
            REPLACE('best', 'b', 'B') AS reverse_best
    """)

print(query)

shape: (1, 2)
┌────────────────┬──────────────┐
│ replace_polars ┆ reverse_best │
│ ---            ┆ ---          │
│ str            ┆ str          │
╞════════════════╪══════════════╡
│ Pola           ┆ Best         │
└────────────────┴──────────────┘


Если заменяемой подстроки в строке нет, **REPLACE()** вернет строку в исходном виде.

In [48]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            REPLACE('Polars', 'qqq', '') AS replace_polars,
            REPLACE('best', 'b', 'B') AS reverse_best
    """)

print(query)

shape: (1, 2)
┌────────────────┬──────────────┐
│ replace_polars ┆ reverse_best │
│ ---            ┆ ---          │
│ str            ┆ str          │
╞════════════════╪══════════════╡
│ Polars         ┆ Best         │
└────────────────┴──────────────┘


##### Функция SUBSTR()

Функция **SUBSTR()** извлекает подстроку из заданной строки, начиная с указанной позиции и заданной длины (длина может быть опущена — тогда извлекается до конца строки). Параметры:
- str - исходная строка,
- start - позиция первого извлекаемого символа,
- len - длина извлекаемой подстроки.

Пример:

In [49]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            SUBSTR('Polars', 3) AS substr_polars1,
            SUBSTR('Polars', 2, 4) AS substr_polars2,
            SUBSTR('Polars', 1, 3) AS substr_polars3
    """)

print(query)

shape: (1, 3)
┌────────────────┬────────────────┬────────────────┐
│ substr_polars1 ┆ substr_polars2 ┆ substr_polars3 │
│ ---            ┆ ---            ┆ ---            │
│ str            ┆ str            ┆ str            │
╞════════════════╪════════════════╪════════════════╡
│ lars           ┆ olar           ┆ Pol            │
└────────────────┴────────────────┴────────────────┘


**Замечание.** Символы исходной строки нумеруются с 1, а не с 0. При указании нуля или отрицательных чисел PolarsSQL мыслено добавляет отступы слева:

In [50]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            SUBSTR('Polars', 0) AS substr_polars0,
            SUBSTR('Polars', -1, 4) AS substr_polars01,
            SUBSTR('Polars', 0, 4) AS substr_polars1,
            SUBSTR('Polars', -1, 4) AS substr_polars2,
            SUBSTR('Polars', -2, 4) AS substr_polars3,
            SUBSTR('Polars', -5, 4) AS substr_polars4, -- будет пустая строка, т.к. длина извлекаемой подстроки по модулю меньше длине отступа
            
    """)

print(query)

shape: (0, 6)
┌────────────────┬────────────────┬────────────────┬───────────────┬───────────────┬───────────────┐
│ substr_polars0 ┆ substr_polars0 ┆ substr_polars1 ┆ substr_polars ┆ substr_polars ┆ substr_polars │
│ ---            ┆ 1              ┆ ---            ┆ 2             ┆ 3             ┆ 4             │
│ str            ┆ ---            ┆ str            ┆ ---           ┆ ---           ┆ ---           │
│                ┆ str            ┆                ┆ str           ┆ str           ┆ str           │
╞════════════════╪════════════════╪════════════════╪═══════════════╪═══════════════╪═══════════════╡
└────────────────┴────────────────┴────────────────┴───────────────┴───────────────┴───────────────┘


Если указанная позиция первого извлекаемого символа превышает длину строки, то получим пустую строку:

In [51]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            SUBSTR('Polars', 0, 4) AS substr_polars1,
            SUBSTR('Polars', 8, 4) AS substr_polars2, 
    """)

print(query)

shape: (1, 2)
┌────────────────┬────────────────┐
│ substr_polars1 ┆ substr_polars2 │
│ ---            ┆ ---            │
│ str            ┆ str            │
╞════════════════╪════════════════╡
│ Pol            ┆                │
└────────────────┴────────────────┘


##### Функция TRIM()

Функция **TRIM()** предназначена для удаления указанной подстроки из начала и/или конца строки. В отличие от других функций, её синтаксис использует ключевое слово FROM, а не запятую для разделения аргументов:
`TRIM(LEADING | TRAILING | BOTH <подстрока> FROM <исходная строка>)`.
- `LEADING` - удаляет подстроку только с начала строки;
- `TRAILING` - удаляет подстроку только с конца строки;
- `BOTH` - удаляет подстроку и с начала, и с конца строки.

Если не указано ни одно из этих ключевых слов, по умолчанию используется `BOTH`.

Пример:

In [52]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            TRIM(LEADING '_' FROM '_Polars_') AS trim_leading,
            TRIM(TRAILING '_' FROM '_Polars_') AS trim_trailing,
            TRIM(BOTH '_' FROM '_Polars_') AS trim_both
    """)

print(query)

shape: (1, 3)
┌──────────────┬───────────────┬───────────┐
│ trim_leading ┆ trim_trailing ┆ trim_both │
│ ---          ┆ ---           ┆ ---       │
│ str          ┆ str           ┆ str       │
╞══════════════╪═══════════════╪═══════════╡
│ Polars_      ┆ _Polars       ┆ Polars    │
└──────────────┴───────────────┴───────────┘


In [53]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT
            TRIM('_' FROM '_Polars_') AS trim_base
    """)

print(query)

shape: (1, 1)
┌───────────┐
│ trim_base │
│ ---       │
│ str       │
╞═══════════╡
│ Polars    │
└───────────┘


Если не указаны и ключевое слово, и удаляемая подстрока, функция **TRIM()** выполнит удаление всех пробелов из начала и конца строки:

In [54]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            TRIM('   Polars ') AS trim_base
    """)

print(query)

shape: (1, 1)
┌───────────┐
│ trim_base │
│ ---       │
│ str       │
╞═══════════╡
│ Polars    │
└───────────┘


##### Примечания.

1. Функции могут применяться не только в **SELECT**, но и в блоках операторов **WHERE** и **ORDER BY**.
2. Функции **CHAR_LENGTH()**, **LOWER()**, **UPPER()**, **LTRIM()**, **RTRIM()**, **REVERSE()**, **LEFT()**, **RIGHT()**, **REPLACE()**, **SUBSTR()**, **TRIM()** при вызове с аргументом NULL вызовут ошибку.
3. Функция **REPLACE()** выполняет замену с учётом регистра.

#### Математические функции

Рассмотрим следующие финкции:
- ABS()
- ROUND()
- POW()
- SQRT()
- PI(), DEGREES(), RADIANS(), SIN(), COS(), TAN()
- LEAST() и GREATEST()

##### Функция ABS()

Функция **ABS()** вычисляет модуль числа: принимает число и возвращает его абсолютное значение.

Пример:

In [55]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            ABS(-1) AS abs_1,
            ABS(-5) AS abs_2,
            ABS(0) AS abs_3,
            ABS(45) AS abs_4
    """)

print(query)

shape: (1, 4)
┌───────┬───────┬───────┬───────┐
│ abs_1 ┆ abs_2 ┆ abs_3 ┆ abs_4 │
│ ---   ┆ ---   ┆ ---   ┆ ---   │
│ i32   ┆ i32   ┆ i32   ┆ i32   │
╞═══════╪═══════╪═══════╪═══════╡
│ 1     ┆ 5     ┆ 0     ┆ 45    │
└───────┴───────┴───────┴───────┘


##### Функция ROUND()

Функция `ROUND(num, decimals)` округляет число `num` до указанного количества знаков после запятой (`decimals`). При `decimals = 0` возвращает целое значение с плавающей точкой. 

Пример:

In [56]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            ROUND(3.1415926535, 3) AS round_1,
            ROUND(3.1415926535, 2) AS round_2,
            ROUND(3.1415926535, 5) AS round_3,
            ROUND(3.1415926535, 1) AS round_4,
            ROUND(3.1415926535, 0) AS round_5
    """)

print(query)

shape: (1, 5)
┌─────────┬─────────┬─────────┬─────────┬─────────┐
│ round_1 ┆ round_2 ┆ round_3 ┆ round_4 ┆ round_5 │
│ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---     │
│ f64     ┆ f64     ┆ f64     ┆ f64     ┆ f64     │
╞═════════╪═════════╪═════════╪═════════╪═════════╡
│ 3.142   ┆ 3.14    ┆ 3.14159 ┆ 3.1     ┆ 3.0     │
└─────────┴─────────┴─────────┴─────────┴─────────┘


##### Функция POW()

Функция `POW(num, degree)` возводит число `num` в степень `degree`.
Если оба аргумента целые, то результат целый (i32/i64); если хотя бы один аргумент с плавающей точкой - результат дробный (f64).
Поддерживает отрицательные степени.

Пример:

In [57]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            POW(2, 3) AS pow_1,
            POW(0, 1) AS pow_2,
            POW(4, 7) AS pow_3,
            POW(2.0, -1) AS pow_4
    """)

print(query)

shape: (1, 4)
┌───────┬───────┬───────┬───────┐
│ pow_1 ┆ pow_2 ┆ pow_3 ┆ pow_4 │
│ ---   ┆ ---   ┆ ---   ┆ ---   │
│ i32   ┆ i32   ┆ i32   ┆ f64   │
╞═══════╪═══════╪═══════╪═══════╡
│ 8     ┆ 0     ┆ 16384 ┆ 0.5   │
└───────┴───────┴───────┴───────┘


##### Функция SQRT()

Функция **SQRT()** вычисляет квадратный корень из числа.
Аргумент должен быть неотрицательным (если отрицательный, то вернётся NaN); результат имеет тип f64 (даже для целых входных значений).

Пример:

In [58]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            SQRT(4) AS sqrt_1,
            SQRT(9) AS sqrt_2,
            SQRT(0) AS sqrt_3,
            SQRT(-1) AS sqrt_4  
    """)

print(query)

shape: (1, 4)
┌────────┬────────┬────────┬────────┐
│ sqrt_1 ┆ sqrt_2 ┆ sqrt_3 ┆ sqrt_4 │
│ ---    ┆ ---    ┆ ---    ┆ ---    │
│ f64    ┆ f64    ┆ f64    ┆ f64    │
╞════════╪════════╪════════╪════════╡
│ 2.0    ┆ 3.0    ┆ 0.0    ┆ NaN    │
└────────┴────────┴────────┴────────┘


##### Функции PI(), DEGREES(), RADIANS(), SIN(), COS(), TAN()

- Функция **PI()** возвращает значение числа π (пи) с точностью до шести знаков после запятой. Не принимает аргументов.
- Функция **DEGREES()** преобразует угол из радиан в градусы.
- Функция **RADIANS()** преобразует угол из градусов в радианы.
- Функции **SIN()**, **COS()** и **TAN()** вычисляют синус, косинус и тангенс угла, заданного в радианах. Принимают один числовой аргумент.

Пример:

In [59]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            PI()         AS pi,
            DEGREES(1)   AS degrees,
            RADIANS(360) AS radians,
            SIN(4)       AS sin,
            COS(1)       AS cos,
            TAN(10)      AS tan
    """)

print(query)

shape: (1, 6)
┌──────────┬──────────┬──────────┬───────────┬──────────┬──────────┐
│ pi       ┆ degrees  ┆ radians  ┆ sin       ┆ cos      ┆ tan      │
│ ---      ┆ ---      ┆ ---      ┆ ---       ┆ ---      ┆ ---      │
│ f64      ┆ f64      ┆ f64      ┆ f64       ┆ f64      ┆ f64      │
╞══════════╪══════════╪══════════╪═══════════╪══════════╪══════════╡
│ 3.141593 ┆ 57.29578 ┆ 6.283185 ┆ -0.756802 ┆ 0.540302 ┆ 0.648361 │
└──────────┴──────────┴──────────┴───────────┴──────────┴──────────┘


##### Функции LEAST() и GREATEST()

Функции **LEAST()** и **GREATEST()** возвращают наименьшее и наибольше значение среди переданных аргументов соответсвенно.
Обе работают с числовыми типами и возвращают результат того же типа.

Пример:

In [60]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            LEAST(22, 12, 23, 45, 41, 8) AS least_base,
            LEAST(2, 1, 3.0, 5, 4) AS least_float,
            LEAST(100, NULL, 2, 0) AS least_any_null,
            LEAST(NULL, NULL, NULL, NULL) AS least_null,
            LEAST(1) AS least_one,
            GREATEST(22, 12, 23, 45, 41, 8) AS greatest_base,
            GREATEST(2, 1, 3.0, 5, 4) AS greates_float,
            GREATEST(100, NULL, 2, 0) AS greates_any_null,
            GREATEST(NULL, NULL, NULL, NULL) AS greates_null,
            GREATEST(1) AS greates_one
    """)

query

least_base,least_float,least_any_null,least_null,least_one,greatest_base,greates_float,greates_any_null,greates_null,greates_one
i32,f64,i32,null,i32,i32,f64,i32,null,i32
8,1.0,0,null,1,45,5.0,100,null,1


Из это примера важно подчеркнуть следующее:
1. NULL не участвует в сравнении: если остались нормальные значения, функция вернёт минимум/максимум среди них.
2. Если все аргументы NULL, то результат NULL.
3. Наличие хотя бы одного f64 приводит весь результат к типу f64.
4. В классическом SQL функции LEAST/GREATEST требуют ≥2 аргументов, но PolarsSQL допускает один (возвращает его без ошибки).

#### Datetime функции

Рассмотрим следующие финкции:
- DATE, TIME, DATETIME
- EXTRACT()
- DATE_PART()
- STRFTIME()

##### Функциии DATE, TIME, DATETIME

Функции **DATE()**, **TIME()** и **DATETIME()** используются для создания значений даты, времени и даты со временем из строк в формате ISO.
- `DATE('YYYY-MM-DD')` - преобразует строку в значение типа date (только дата).
- `TIME('HH:MM:SS')` - преобразует строку в значение типа time (только время).
- `DATETIME('YYYY-MM-DD HH:MM:SS')` - преобразует строку в значение типа datetime[μs] (дата и время с точностью до микросекунд).

Все функции принимают один строковый аргумент в строгом ISO-формате и возвращают соответствующий тип даты/времени, поддерживаемый Polars.

Пример:

In [61]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            DATE '2025-11-10' AS date,
            TIME '18:51:42' AS time,
            DATETIME '2025-11-10 18:52:45' AS datetime
    """)

print(query)

shape: (1, 3)
┌────────────┬──────────┬─────────────────────┐
│ date       ┆ time     ┆ datetime            │
│ ---        ┆ ---      ┆ ---                 │
│ date       ┆ time     ┆ datetime[μs]        │
╞════════════╪══════════╪═════════════════════╡
│ 2025-11-10 ┆ 18:51:42 ┆ 2025-11-10 18:52:45 │
└────────────┴──────────┴─────────────────────┘


**Замечание.** Данные функции можно писать как в скобках, так и без них.

##### Функция EXTRACT()

Функция `EXTRACT(part FROM date/datetime)` извлекает указанную часть даты или времени (например, год, месяц, день недели и т.д.) из значения типа date или datetime.

- Первый аргумент - название части (например, 'year', 'month', 'hour', 'week' и др.), указывается без кавычек в SQL-запросе.
- Второй аргумент - столбец или выражение с датой/временем.
- Результат - целое число (или строка для некоторых частей, например timezone), соответствующее запрошенной временной компоненте.

Пример:

In [62]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            EXTRACT(YEAR FROM DATE '2025-11-10') AS year,
            EXTRACT(MONTH FROM DATE '2025-11-10') AS month,
            EXTRACT(DAY FROM DATE '2025-11-10') AS day,
            EXTRACT(WEEK FROM DATE '2025-11-10') AS week,
            EXTRACT(QUARTER FROM DATE '2025-11-10') AS quarter,
    """)

print(query)

shape: (1, 5)
┌──────┬───────┬─────┬──────┬─────────┐
│ year ┆ month ┆ day ┆ week ┆ quarter │
│ ---  ┆ ---   ┆ --- ┆ ---  ┆ ---     │
│ i32  ┆ i8    ┆ i8  ┆ i8   ┆ i8      │
╞══════╪═══════╪═════╪══════╪═════════╡
│ 2025 ┆ 11    ┆ 10  ┆ 46   ┆ 4       │
└──────┴───────┴─────┴──────┴─────────┘


##### Функция DATE_PART()

Функция `DATE_PART(part, date/datetime)` извлекает указанную временную компоненту (например, год, месяц, час и т.д.) из значения типа date или datetime.
- Первый аргумент - название части в виде строки в кавычках (например, 'year', 'hour', 'week').
- Второй аргумент - столбец или выражение с датой/временем.
- Возвращает целое число (или i64 для epoch), соответствующее запрашиваемой части.

Пример:

In [63]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            DATE_PART('YEAR', DATE '2025-11-10')       AS year,
            DATE_PART('MONTH', DATE '2025-11-10')      AS month,
            DATE_PART('DAY', DATE '2025-11-10')        AS day,
            DATE_PART('WEEK', DATE '2025-11-10')       AS week,
            DATE_PART('QUARTER', DATE '2025-11-10')    AS quarter,
            DATE_PART('HOUR', TIME '18:51:42')         AS hour,
    """)

print(query)

shape: (1, 6)
┌──────┬───────┬─────┬──────┬─────────┬──────┐
│ year ┆ month ┆ day ┆ week ┆ quarter ┆ hour │
│ ---  ┆ ---   ┆ --- ┆ ---  ┆ ---     ┆ ---  │
│ i32  ┆ i8    ┆ i8  ┆ i8   ┆ i8      ┆ i8   │
╞══════╪═══════╪═════╪══════╪═════════╪══════╡
│ 2025 ┆ 11    ┆ 10  ┆ 46   ┆ 4       ┆ 18   │
└──────┴───────┴─────┴──────┴─────────┴──────┘


**DATE_PART()** - это альтернатива функции **EXTRACT()**, но с другим синтаксисом.
Обе функции поддерживают одинаковый набор временных частей (год, квартал, день недели, эпоха, наносекунды и др.).

##### Функция STRFTIME()

Функция `STRFTIME(temporal_value, format_string)` преобразует значение даты, времени или даты со временем (Date, Time, Datetime) в строку, отформатированную по заданному шаблону.

- Первый аргумент - столбец или выражение с типом даты/времени.
- Второй аргумент - строка формата в стиле strftime.
- Если входное значение NULL то, результат также будет NULL.
- Результат всегда имеет тип str.

Примеры форматов:
- `'%Y-%m-%d'` → 2025-11-10
- `'%B %d, %Y'` → November 10, 2025 (месяц словом)
- `'%H.%M.%S'` → 18.30.45 (время с точками как разделителями)
- `'%A'` → Monday (полное название дня недели)

**Замечание.** Формат чувствителен к регистру: `%Y` - год из 4 цифр, `%y` - из 2. 

Пример:

In [64]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            DATE '2025-11-11' AS dt
            ,STRFTIME(DATE '2025-11-11', '%d.%m.%Y') AS s_dt_1
            ,STRFTIME(DATETIME '2025-11-11 00:12:54', '%r') AS s_dt_2
    """)

print(query)

shape: (1, 3)
┌────────────┬────────────┬─────────────┐
│ dt         ┆ s_dt_1     ┆ s_dt_2      │
│ ---        ┆ ---        ┆ ---         │
│ date       ┆ str        ┆ str         │
╞════════════╪════════════╪═════════════╡
│ 2025-11-11 ┆ 11.11.2025 ┆ 12:12:54 AM │
└────────────┴────────────┴─────────────┘


Со всеми поддерживаемыми форматами можно ознакомиться по [ссылке](https://docs.rs/chrono/latest/chrono/format/strftime/).

#### Дополнительные функции

Рассмотрим следующие функции:
- CAST()
- TRY_CAST()
- COALESCE()
- IF()
- IFNULL()
- NULLIF()

##### Функция CAST()

Функция `CAST(value AS target_type)` (или сокращённо `value::target_type`) преобразует значение к указанному типу данных.

Если преобразование невозможно (например, отрицательное число в беззнаковый тип, или некорректная дата), запрос завершится ошибкой.
Используется, когда вы уверены, что данные корректны.

Пример:

In [65]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            CAST(5.4 AS int64)                  AS cast_1,
            5::float4                           AS cast_2,
            '2025-11-11'::date                  AS cast_3,
            CAST(DATE('2025-11-11') AS string)  AS cast_4
            
    """)

print(query)

shape: (1, 4)
┌────────┬────────┬────────────┬────────────┐
│ cast_1 ┆ cast_2 ┆ cast_3     ┆ cast_4     │
│ ---    ┆ ---    ┆ ---        ┆ ---        │
│ i64    ┆ f32    ┆ date       ┆ str        │
╞════════╪════════╪════════════╪════════════╡
│ 5      ┆ 5.0    ┆ 2025-11-11 ┆ 2025-11-11 │
└────────┴────────┴────────────┴────────────┘


##### Функция TRY_CAST()

Функция `TRY_CAST(value AS target_type)` пытается преобразовать значение к указанному типу, но не вызывает ошибку при неудаче.

Если преобразование невозможно, возвращается NULL.
Идеально подходит для «грязных» данных (например, даты вроде "N/A" или числа за пределами диапазона).

Пример:

In [66]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            TRY_CAST('polars' AS int64)  AS try_cast_1, -- строку в целочисленное число
            TRY_CAST('polars' AS date)   AS try_cast_2, -- строку в дату
            TRY_CAST(5.4 AS int64)       AS try_cast_3
    """)

print(query)

shape: (1, 3)
┌────────────┬────────────┬────────────┐
│ try_cast_1 ┆ try_cast_2 ┆ try_cast_3 │
│ ---        ┆ ---        ┆ ---        │
│ i64        ┆ date       ┆ i64        │
╞════════════╪════════════╪════════════╡
│ null       ┆ null       ┆ 5          │
└────────────┴────────────┴────────────┘


В отличие от **CAST()**, **TRY_CAST()** никогда не прервёт выполнение запроса - вместо ошибки вы получите NULL. 

##### Функция COALESCE()

Функция **COALESCE()** возвращает первое значение из списка, которое не равно NULL.

Принимает переменное количество аргументов (столбцов или выражений).
Если все аргументы равны NULL, то результат NULL.
Используется для замены пропущенных значений альтернативным значением или для последовательного выбора из нескольких столбцов.

Пример:

In [67]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            COALESCE('polars', NULL, 'pandas')         AS coalesce_1,
            COALESCE(NULL, 'выбери меня', 'нет, меня') AS coalesce_2,
            COALESCE(NULL, NULL, NULL)                 AS coalesce_null
    """)

print(query)

shape: (1, 3)
┌────────────┬─────────────┬───────────────┐
│ coalesce_1 ┆ coalesce_2  ┆ coalesce_null │
│ ---        ┆ ---         ┆ ---           │
│ str        ┆ str         ┆ null          │
╞════════════╪═════════════╪═══════════════╡
│ polars     ┆ выбери меня ┆ null          │
└────────────┴─────────────┴───────────────┘


##### Функция IF()

Функция `IF(condition, expr1, expr2)` возвращает `expr1`, если условие `condition` истинно, и `expr2`, если условие ложно.

- `condition` - логическое выражение.
- `expr1` - значение, возвращаемое при истине.
- `expr2` - значение, возвращаемое при лжи.

Пример:

In [68]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            IF(100 < 1000, 'меньше 1000', 'больше 1000')   AS if_1,
            IF(1000 < 1000, 'меньше 1000', 'больше 1000')  AS if_2,
            IF(NULL < 1000, 'меньше 1000', 'больше 1000')  AS if_3,
    """)

print(query)

shape: (1, 3)
┌─────────────┬─────────────┬─────────────┐
│ if_1        ┆ if_2        ┆ if_3        │
│ ---         ┆ ---         ┆ ---         │
│ str         ┆ str         ┆ str         │
╞═════════════╪═════════════╪═════════════╡
│ меньше 1000 ┆ больше 1000 ┆ больше 1000 │
└─────────────┴─────────────┴─────────────┘


##### Функция IFNULL()

Функция `IFNULL(value, alternative)` проверяет, равно ли `value` значению NULL.
- Если `value` есть NULL, то возвращается `alternative`.
- Если `value` не NULL,  то возвращается `value`.

Используется для замены пропущенных значений на заданное по умолчанию.

Пример:

In [69]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            IFNULL(NULL, 'замени меня')  AS ifnull_1,
            IFNULL(1000, 'замени меня')  AS ifnull_2,
    """)

print(query)

shape: (1, 2)
┌─────────────┬──────────┐
│ ifnull_1    ┆ ifnull_2 │
│ ---         ┆ ---      │
│ str         ┆ str      │
╞═════════════╪══════════╡
│ замени меня ┆ 1000     │
└─────────────┴──────────┘


##### Функция NULLIF()

Функция `NULLIF(expr1, expr2)` сравнивает два выражения:
- Если `expr1` равно `expr2`, то возвращается NULL.
- Если `expr1` не равно `expr2`, то возвращается `expr1`.

Пример:

In [70]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            NULLIF(30, 30)          AS nullif_1,
            NULLIF(1000, 1)         AS nullif_2,
            NULLIF(NULL, NULL)      AS nullif_3,
            NULLIF(NULL, 7)         AS nullif_4,
            NULLIF('polars', NULL)  AS nullif_5
    """)

print(query)

shape: (1, 5)
┌──────────┬──────────┬──────────┬──────────┬──────────┐
│ nullif_1 ┆ nullif_2 ┆ nullif_3 ┆ nullif_4 ┆ nullif_5 │
│ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ i32      ┆ i32      ┆ null     ┆ null     ┆ str      │
╞══════════╪══════════╪══════════╪══════════╪══════════╡
│ null     ┆ 1000     ┆ null     ┆ null     ┆ polars   │
└──────────┴──────────┴──────────┴──────────┴──────────┘


#### Условное ветвление

Конструкция с оператором **CASE** позволяет выполнять условные проверки и возвращать разные значения в зависимости от выполнения условий. Работает по принципу «если ..., то ..., иначе если ..., то ...».

Синтаксис: 
```
CASE
  WHEN condition1 THEN result1
  WHEN condition2 THEN result2
  ...
  ELSE default_result
END
```
Проверяет условия по порядку, сверху вниз.
Возвращает первое значение, для которого условие истинно (true).
Если ни одно условие не выполнено, возвращается значение из ELSE.
Если ELSE нет, а все условия ложны, то результат будет NULL.

Пример:

In [71]:
with pl.SQLContext(t=df_t, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            100 AS value,
            CASE
                WHEN 100 < 10 THEN 'мало'
                WHEN 100 = 100 THEN 'нормально'
                ELSE 'много'
            END AS category,
            CASE 
                WHEN 100 < 1 THEN 'очень мало'
                WHEN 100 < 10 THEN 'мало'
                ELSE 'нормально'
            END AS category_2
    """)

print(query)

shape: (1, 3)
┌───────┬───────────┬────────────┐
│ value ┆ category  ┆ category_2 │
│ ---   ┆ ---       ┆ ---        │
│ i32   ┆ str       ┆ str        │
╞═══════╪═══════════╪════════════╡
│ 100   ┆ нормально ┆ нормально  │
└───────┴───────────┴────────────┘


Данная функция полезна для классификации данных или замены значений по правилам.
Конструкция с оператором **CASE** гибче функции **IF()**, так как может учитывать множественное ветвление или выстроить последовательную проверку условий.

### Группировка и агрегация данных

#### Агрегатные функции

При анализе данных редко достаточно просто просмотреть исходные записи. Часто требуется получить обобщённую информацию о наборе данных: какова общая сумма? Какое значение максимальное? Сколько записей удовлетворяют условию?

Для этого в PolarsSQL используются агрегатные функции. Это специальные функции, которые выполняют вычисления на группе значений и возвращают одно результирующее значение.

Примеры агрегатных функций:
- **SUM()** — сумма всех значений;
- **AVG()** — среднее арифметическое;
- **MEDIAN()** — медианное значение;
- **COUNT()** — количество записей;
- **MIN()** — минимальное значение;
- **MAX()** — максимальное значение;

Эти функции особенно полезны при построении аналитических отчётов, дэшбордов.

##### Функция SUM()

Функция **SUM()** вычисляет сумму всех числовых значений в указанном столбце. Она применяется для получения итоговых показателей, таких как общая выручка, суммарное количество событий или совокупное значение метрики. Например, с помощью **SUM()** можно определить общий вес всех бриллиантов в наборе данных.

In [72]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            SUM(carat) AS total_carat
        FROM diamonds
    """)

print(query)

shape: (1, 1)
┌─────────────┐
│ total_carat │
│ ---         │
│ f64         │
╞═════════════╡
│ 43040.87    │
└─────────────┘


Или же найдём вес камней огранки `'Premium'`:

In [73]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            SUM(carat) AS total_carat
        FROM diamonds
        WHERE cut='Premium'
    """)

print(query)

shape: (1, 1)
┌─────────────┐
│ total_carat │
│ ---         │
│ f64         │
╞═════════════╡
│ 12300.95    │
└─────────────┘


**Примечание.** Функция **SUM()** значения NULL интерпретирует как ноль.

In [74]:
# DataFrame с NULL 
df_test_null = pl.DataFrame({"val": (5, 6, 2, 4, None)})

print(df_test_null)

shape: (5, 1)
┌──────┐
│ val  │
│ ---  │
│ i64  │
╞══════╡
│ 5    │
│ 6    │
│ 2    │
│ 4    │
│ null │
└──────┘


Посчитаем сумму данных значений:

In [75]:
with pl.SQLContext(t_null=df_test_null, eager=True) as ctx:
    query = ctx.execute("""
       SELECT SUM(val) AS total
       FROM t_null
    """)

print(query)

shape: (1, 1)
┌───────┐
│ total │
│ ---   │
│ i64   │
╞═══════╡
│ 17    │
└───────┘


А теперь только NULL значение:

In [76]:
with pl.SQLContext(t_null=df_test_null, eager=True) as ctx:
    query = ctx.execute("""
       SELECT SUM(val) AS total
       FROM t_null
       WHERE val IS NULL
    """)

print(query)

shape: (1, 1)
┌───────┐
│ total │
│ ---   │
│ i64   │
╞═══════╡
│ 0     │
└───────┘


##### Функция AVG()

Функция **AVG()** используется для вычисления среднего арифметического всех числовых значений в указанном столбце. Например, найдём среднюю стоимость всех камней:

In [77]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            AVG(price) AS avg_price
        FROM diamonds
    """)

print(query)

shape: (1, 1)
┌─────────────┐
│ avg_price   │
│ ---         │
│ f64         │
╞═════════════╡
│ 3932.799722 │
└─────────────┘


Функция **AVG()** игнорирует NULL значения.

In [78]:
with pl.SQLContext(t_null=df_test_null, eager=True) as ctx:
    query = ctx.execute("""
       SELECT AVG(val) AS avg_val
       FROM t_null
    """)

print(query)

shape: (1, 1)
┌─────────┐
│ avg_val │
│ ---     │
│ f64     │
╞═════════╡
│ 4.25    │
└─────────┘


In [79]:
with pl.SQLContext(t_null=df_test_null, eager=True) as ctx:
    query = ctx.execute("""
       SELECT AVG(val) AS avg_val
       FROM t_null
       WHERE val IS NULL
    """)

print(query)

shape: (1, 1)
┌─────────┐
│ avg_val │
│ ---     │
│ f64     │
╞═════════╡
│ null    │
└─────────┘


##### Функция COUNT()

Функция **COUNT()** используется для подсчёта количества строк в результате запроса.
- При вызове с аргументом `*` (`COUNT(*)`) функция возвращает общее число записей в таблице, включая строки с NULL.
- При вызове с указанием столбца (`COUNT(column)`) функция подсчитывает только непустые значения (не равные NULL) в этом столбце.

In [80]:
with pl.SQLContext(t_null=df_test_null, eager=True) as ctx:
    query = ctx.execute("""
       SELECT COUNT(*) AS cnt
       FROM t_null
    """)

print(query)

shape: (1, 1)
┌─────┐
│ cnt │
│ --- │
│ u32 │
╞═════╡
│ 5   │
└─────┘


In [81]:
with pl.SQLContext(t_null=df_test_null, eager=True) as ctx:
    query = ctx.execute("""
       SELECT COUNT(val) AS cnt
       FROM t_null
    """)

print(query)

shape: (1, 1)
┌─────┐
│ cnt │
│ --- │
│ u32 │
╞═════╡
│ 4   │
└─────┘


Таким образом, **COUNT(*)** применяется для определения размера набора данных, а **COUNT(column)** - для анализа заполненности конкретного поля.

##### Функции MIN() и MAX()

Функции **MIN()** и **MAX()** возвращают наименьшее и наибольшее значение соответственно из указанного столбца. Они применяются для определения границ диапазона данных. Например, минимальной или максимальной продолжительности, самой ранней даты, наибольшей суммы продаж. Так, с помощью **MIN()** можем найти наименьшую стоимость камня в наборе, а с помощью **MAX()** - самого высокого.

In [82]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            MIN(price) AS min_price,
            MAX(price) AS max_price
        FROM diamonds
    """)

print(query)

shape: (1, 2)
┌───────────┬───────────┐
│ min_price ┆ max_price │
│ ---       ┆ ---       │
│ i64       ┆ i64       │
╞═══════════╪═══════════╡
│ 326       ┆ 18823     │
└───────────┴───────────┘


Функции **MIN()** и **MAX()** игнорируют NULL значения.

In [83]:
with pl.SQLContext(t_null=df_test_null, eager=True) as ctx:
    query = ctx.execute("""
       SELECT MIN(val) AS avg_val
       FROM t_null
       WHERE val IS NULL
    """)

print(query)

shape: (1, 1)
┌─────────┐
│ avg_val │
│ ---     │
│ i64     │
╞═════════╡
│ null    │
└─────────┘


##### **Примечания.**

**Примечание 1.** Внутри функций **COUNT()** можно использовать ключевое слово **DISTINCT**, чтобы в итоговых вычислениях участвовали лишь уникальные значения поля, т.е. посчитаем количество уникальных элементов в столбце.

In [84]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            COUNT(DISTINCT cut) AS cut_uniq
        FROM diamonds
    """)

print(query)

shape: (1, 1)
┌──────────┐
│ cut_uniq │
│ ---      │
│ u32      │
╞══════════╡
│ 5        │
└──────────┘


**Примечание 2.** Если все значения поля, переданного в качестве аргумента в агрегатную функцию, имеют значение NULL, возвращаемым значением функции также будет значение NULL. Исключением является функции **COUNT()** и **SUM()**, которые вернут значение 0.

In [85]:
with pl.SQLContext(t_null=df_test_null, eager=True) as ctx:
    query = ctx.execute("""
       SELECT 
           COUNT(val) AS count,
           SUM(val) AS sum,
           AVG(val) AS avg,
           MAX(val) AS max,
           MIN(val) AS min,
           MEDIAN(val) AS median
       FROM t_null
       WHERE val IS NULL
    """)

print(query)

shape: (1, 6)
┌───────┬─────┬──────┬──────┬──────┬────────┐
│ count ┆ sum ┆ avg  ┆ max  ┆ min  ┆ median │
│ ---   ┆ --- ┆ ---  ┆ ---  ┆ ---  ┆ ---    │
│ u32   ┆ i64 ┆ f64  ┆ i64  ┆ i64  ┆ f64    │
╞═══════╪═════╪══════╪══════╪══════╪════════╡
│ 0     ┆ 0   ┆ null ┆ null ┆ null ┆ null   │
└───────┴─────┴──────┴──────┴──────┴────────┘


**Примечание 3.** PolarsSQL позволяет агрегатные функции (**AVG**, **MAX**, **MIN**, **SUM**, **MEDIAN**) использовать в блоке **WHERE**!

In [86]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT *
        FROM diamonds
        WHERE price = MEDIAN(price) 
    """)

print(query)

shape: (26, 11)
┌───────┬───────┬───────────┬───────┬───┬───────┬──────┬──────┬──────┐
│ id    ┆ carat ┆ cut       ┆ color ┆ … ┆ price ┆ x    ┆ y    ┆ z    │
│ ---   ┆ ---   ┆ ---       ┆ ---   ┆   ┆ ---   ┆ ---  ┆ ---  ┆ ---  │
│ i64   ┆ f64   ┆ str       ┆ str   ┆   ┆ i64   ┆ f64  ┆ f64  ┆ f64  │
╞═══════╪═══════╪═══════════╪═══════╪═══╪═══════╪══════╪══════╪══════╡
│ 51710 ┆ 0.58  ┆ Ideal     ┆ F     ┆ … ┆ 2401  ┆ 5.39 ┆ 5.35 ┆ 3.32 │
│ 51711 ┆ 0.72  ┆ Very Good ┆ I     ┆ … ┆ 2401  ┆ 5.78 ┆ 5.82 ┆ 3.52 │
│ 51712 ┆ 0.7   ┆ Very Good ┆ F     ┆ … ┆ 2401  ┆ 5.58 ┆ 5.66 ┆ 3.57 │
│ 51713 ┆ 0.78  ┆ Very Good ┆ G     ┆ … ┆ 2401  ┆ 5.82 ┆ 5.85 ┆ 3.72 │
│ 51714 ┆ 0.72  ┆ Very Good ┆ E     ┆ … ┆ 2401  ┆ 5.72 ┆ 5.79 ┆ 3.58 │
│ …     ┆ …     ┆ …         ┆ …     ┆ … ┆ …     ┆ …    ┆ …    ┆ …    │
│ 51731 ┆ 0.73  ┆ Fair      ┆ G     ┆ … ┆ 2401  ┆ 5.91 ┆ 5.81 ┆ 3.52 │
│ 51732 ┆ 0.71  ┆ Fair      ┆ H     ┆ … ┆ 2401  ┆ 5.54 ┆ 5.49 ┆ 3.66 │
│ 51733 ┆ 0.71  ┆ Premium   ┆ H     ┆ … ┆ 2401  ┆ 5.81 ┆ 5.67

Почему это неожиданно?
Потому что в стандартном SQL (PostgreSQL, MySQL, ORACLE, etc.) - это ошибка: `Aggregate functions are not allowed in WHERE` или `ORA-00934 group function is not allowed here`.

PolarsSQL - это не стандартный SQL-движок. Он анализирует запрос на уровне плана выполнения и перестраивает его. Это не баг - это фича!

#### Группировка данных

Чтобы получить обобщённую информацию по определённым группам используют группировки - механизм, позволяющий разделить набор данных на логические группы на основе значений одного или нескольких столбцов. Внутри каждой группы можно выполнить агрегатные вычисления, такие как подсчёт, суммирование, поиск минимума/максимума и т.д.

Группировка в PolarsSQL реализуется с помощью оператора **GROUP BY**, после которого указывается один или несколько столбцов. Все строки с одинаковым значением в этих столбцах объединяются в одну группу.

Например, при группировке по столбцу `cut` все бриллианты одной огранки попадают в одну группу. Затем к каждой группе независимо применяются агрегатные функции.

In [87]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            cut,
            COUNT(*) AS cnt_rows
        FROM diamonds
        GROUP BY cut
    """)

print(query)

shape: (5, 2)
┌───────────┬──────────┐
│ cut       ┆ cnt_rows │
│ ---       ┆ ---      │
│ str       ┆ u32      │
╞═══════════╪══════════╡
│ Premium   ┆ 13791    │
│ Very Good ┆ 12082    │
│ Good      ┆ 4906     │
│ Ideal     ┆ 21551    │
│ Fair      ┆ 1610     │
└───────────┴──────────┘


Разбор запроса:
1. Данные разбиваются на группы по уникальным значениям поля `cut`.
2. Для каждой группы выполняется расчёт с помощью `COUNT(*)` - число строк в группе.
3. Результат - одна строка на каждый уникальный `cut`, содержащая наименование огранки и количество бриллиантов.

Иногда одной колонки недостаточно, чтобы получить нужную группировку. Например, хотим узнать количество бриллиантов различного цвета от качества огранки.
Для таких случаев PolarsSQL поддерживает группировку по нескольким полям.

In [88]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            cut, color,
            COUNT(*) AS cnt_rows
        FROM diamonds
        GROUP BY cut, color
    """)

print(query)

shape: (35, 3)
┌───────────┬───────┬──────────┐
│ cut       ┆ color ┆ cnt_rows │
│ ---       ┆ ---   ┆ ---      │
│ str       ┆ str   ┆ u32      │
╞═══════════╪═══════╪══════════╡
│ Ideal     ┆ E     ┆ 3903     │
│ Very Good ┆ E     ┆ 2400     │
│ Fair      ┆ G     ┆ 314      │
│ Premium   ┆ G     ┆ 2924     │
│ Good      ┆ I     ┆ 522      │
│ …         ┆ …     ┆ …        │
│ Premium   ┆ E     ┆ 2337     │
│ Good      ┆ G     ┆ 871      │
│ Good      ┆ D     ┆ 662      │
│ Very Good ┆ D     ┆ 1513     │
│ Ideal     ┆ D     ┆ 2834     │
└───────────┴───────┴──────────┘


**Примечания.**
1. Группировку можно производить по вычисляемому полю.
2. При применении группировки с оператором **GROUP BY** контекст запроса изменяется: вместо работы с отдельными строками мы работаем с логическими группами записей. Это накладывает важные ограничения на то, какие данные можно извлекать в блоке **SELECT**.
    - Нельзя делать:
        - Извлекать столбцы, не входящие в **GROUP BY**, если они не используются в агрегатных функциях;
        - Сортировать по полям, не участвующим в группировке или агрегации;
    - Можно использовать:
        - Поля из **GROUP BY**;
        - Агрегатные функции;
        - Константы - фиксированные значения, одинаковые для всех строк.

Поля в **GROUP BY** позволяют идентифицировать каждую группу.
Агрегатные функции дают возможность выполнять расчёты по группе: подсчитывать, суммировать, находить экстремумы.
Вместе они формируют осмысленный итоговый результат.

#### Фильтрация групп

Для фильтрации групп используется оператор **HAVING** - он применяется после **GROUP BY** и отбирает группы по условию, в отличие от **WHERE**, который фильтрует отдельные строки.

Пример:

In [89]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            cut,
            COUNT(*) AS cnt_rows
        FROM diamonds
        GROUP BY cut
        HAVING cnt_rows >= 12000
    """)

print(query)

shape: (3, 2)
┌───────────┬──────────┐
│ cut       ┆ cnt_rows │
│ ---       ┆ ---      │
│ str       ┆ u32      │
╞═══════════╪══════════╡
│ Ideal     ┆ 21551    │
│ Very Good ┆ 12082    │
│ Premium   ┆ 13791    │
└───────────┴──────────┘


Отличительная особенность PolarsSQL по сравнению с классическим SQL в том, что в фильтрации групп мы должны использовать алиасы агрегатных функций: агрегатная функция должна быть сначала указана в **SELECT** с псевдонимом, а затем этот псевдоним можно использовать в **HAVING**.

Это отличается от классического SQL, где в **HAVING** можно использовать агрегатные функции напрямую (например, `HAVING COUNT(*) >= 12000`). В PolarsSQL такой подход может привести к неожиданным результатам. Использование алиасов в **HAVING** обеспечивает корректную фильтрацию групп.

Сгруппированные данные при извлечении могут быть не только отфильтрованы, но и отсортированы. Для сортировки групп используется оператор **ORDER BY**, который применяется после **GROUP BY** и **HAVING**.

In [90]:
with pl.SQLContext(diamonds=df, eager=True) as ctx:
    query = ctx.execute("""
        SELECT 
            cut,
            COUNT(*) AS cnt_rows
        FROM diamonds
        GROUP BY cut
        HAVING cnt_rows >= 12000
        ORDER BY cnt_rows DESC
    """)

print(query)

shape: (3, 2)
┌───────────┬──────────┐
│ cut       ┆ cnt_rows │
│ ---       ┆ ---      │
│ str       ┆ u32      │
╞═══════════╪══════════╡
│ Ideal     ┆ 21551    │
│ Premium   ┆ 13791    │
│ Very Good ┆ 12082    │
└───────────┴──────────┘


При написании SQL-запроса важно соблюдать не только логическую, но и синтаксическую последовательность операторов. Каждый оператор выполняется в строго определённом порядке, что влияет на результат запроса.

Ниже приведены основные операторы в правильном порядке их размещения в запросе:
1. **SELECT** - Указывает, какие столбцы и агрегаты нужно извлечь в результат.
2. **FROM** - Определяет источник данных.
3. **WHERE** - Фильтрует строки до группировки (на уровне отдельных записей).
4. **GROUP BY** - Группирует строки по значениям одного или нескольких столбцов.
5. **HAVING** - Фильтрует сгруппированные данные по условиям на агрегаты.
6. **ORDER BY** - Сортирует итоговые строки результата.
7. **LIMIT** - Ограничивает количество возвращаемых строк.

### Операции над множествами

**Операции над множествами** — это способ объединения результатов двух или более SQL-запросов в один результирующий набор. Эти операции работают по правилам теории множеств и позволяют:
- объединять данные,
- находить пересечения,
- исключать дубликаты,
- вычитать наборы.

В PolarsSQL доступны следующие операции:
- **UNION**
- **UNION ALL**
- **UNION BY NAME**
- **INTERSECT**
- **EXCEPT**

#### UNION

**UNION** объединяет результаты двух или более запросов, удаляя дубликаты. Каждая строка в итоговом результате уникальна.
Должны соблюдаться следующие требования:
1. Число и порядок столбцов в обоих запросах должны совпадать.
2. Типы данных столбцов должны быть совместимы.

Пример:

In [91]:
# Два множества
df_set1 = pl.DataFrame({"value":[1, 2, 5, 8, 10]})
df_set2 = pl.DataFrame({"value":[1, 2, 3, 4, 5, 9]})

with pl.SQLContext(set1=df_set1, set2=df_set2, eager=True) as ctx:
    query = ctx.execute("""
        SELECT value FROM set1
        UNION
        SELECT value FROM set2
    """)

print(query)

shape: (8, 1)
┌───────┐
│ value │
│ ---   │
│ i64   │
╞═══════╡
│ 10    │
│ 9     │
│ 2     │
│ 5     │
│ 3     │
│ 8     │
│ 4     │
│ 1     │
└───────┘


Данный запрос вернул список уникальных значений, которые встречаются в любом из запросов.

#### UNION ALL

**UNION ALL** объединяет результаты всех строк из обоих запросов без удаления дубликатов. Работает быстрее, чем **UNION**, так как не тратит время на поиск и удаление дублей.

Пример:

In [92]:
# Два множества
df_set1 = pl.DataFrame({"value":[1, 2, 5, 8, 10]})
df_set2 = pl.DataFrame({"value":[1, 2, 3, 4, 5, 9]})

with pl.SQLContext(set1=df_set1, set2=df_set2, eager=True) as ctx:
    query = ctx.execute("""
        SELECT value FROM set1
        UNION ALL
        SELECT value FROM set2
    """)

print(query)

shape: (11, 1)
┌───────┐
│ value │
│ ---   │
│ i64   │
╞═══════╡
│ 1     │
│ 2     │
│ 5     │
│ 8     │
│ 10    │
│ …     │
│ 2     │
│ 3     │
│ 4     │
│ 5     │
│ 9     │
└───────┘


#### UNION BY NAME

**UNION BY NAME** объединяет таблицы по именам столбцов, а не по их порядку. Если в одном запросе есть столбец, которого нет в другом, то он добавляется, и в соответствующих строках будет null.

Пример:

In [93]:
# Два множества
df_set1 = pl.DataFrame({"value":[1, 2, 5, 8, 10]})
df_set3 = pl.DataFrame({"value":[1, 2, 3, 4, 5, 9], "pos":[21, 22, 23, 24, 25, 26]})

with pl.SQLContext(set1=df_set1, set3=df_set3, eager=True) as ctx:
    query = ctx.execute("""
        SELECT * FROM set1
        UNION BY NAME
        SELECT * FROM set3
    """)

print(query)

shape: (11, 2)
┌───────┬──────┐
│ value ┆ pos  │
│ ---   ┆ ---  │
│ i64   ┆ i64  │
╞═══════╪══════╡
│ 5     ┆ 25   │
│ 3     ┆ 23   │
│ 4     ┆ 24   │
│ 9     ┆ 26   │
│ 1     ┆ null │
│ …     ┆ …    │
│ 10    ┆ null │
│ 2     ┆ null │
│ 8     ┆ null │
│ 1     ┆ 21   │
│ 5     ┆ null │
└───────┴──────┘


#### INTERSECT

**INTERSECT** используется для получения пересечения двух или более наборов данных. Он возвращает только те строки, которые присутствуют в обоих запросах.

Пример:

In [94]:
# Два множества
df_set1 = pl.DataFrame({"value":[1, 2, 5, 8, 10]})
df_set2 = pl.DataFrame({"value":[1, 2, 3, 4, 5, 9]})

with pl.SQLContext(set1=df_set1, set2=df_set2, eager=True) as ctx:
    query = ctx.execute("""
        SELECT value FROM set1
        INTERSECT
        SELECT value FROM set2
    """)

print(query)

shape: (3, 1)
┌───────┐
│ value │
│ ---   │
│ i64   │
╞═══════╡
│ 2     │
│ 1     │
│ 5     │
└───────┘


#### EXCEPT

**EXCEPT** возвращает строки из первого запроса, которые не встречаются во втором запросе. То есть из первого множества исключаем элементы, которые входят и во второе множество.

In [95]:
# Два множества
df_set1 = pl.DataFrame({"value":[1, 2, 5, 8, 10]})
df_set2 = pl.DataFrame({"value":[1, 2, 3, 4, 5, 9]})

with pl.SQLContext(set1=df_set1, set2=df_set2, eager=True) as ctx:
    query = ctx.execute("""
        SELECT value FROM set1
        EXCEPT
        SELECT value FROM set2
    """)

print(query)

shape: (2, 1)
┌───────┐
│ value │
│ ---   │
│ i64   │
╞═══════╡
│ 10    │
│ 8     │
└───────┘


### Операции соединений

**JOIN** — это операция в PolarsSQL, которая объединяет строки из двух или более таблиц на основе связанного (совпадающего) столбца.
Она позволяет комбинировать данные из разных таблиц в один результат, что особенно полезно при работе с нормализованными данными.

Виды соединений:
1. INNER JOIN
2. LEFT JOIN (LEFT OUTER JOIN)
3. RIGHT JOIN (RIGHT OUTER JOIN)
4. FULL JOIN (FULL OUTER JOIN)
5. CROSS JOIN
6. NATURAL JOIN
7. SEMI JOIN

#### INNER JOIN

Соединение **INNER JOIN** возвращает только те строки, у которых есть совпадения в обеих таблицах.
Если строка из левой таблицы не имеет соответствия в правой, или наоборот - она не попадает в результат.

Пример:

In [96]:
df1 = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"]
})

df2 = pl.DataFrame({
    "id": [2, 3, 4],
    "age": [25, 30, 35]
})

with pl.SQLContext(df1=df1, df2=df2, eager=True) as ctx:
    result = ctx.execute("""
        SELECT df1.id, df1.name, df2.age
        FROM df1
            INNER JOIN df2 ON df1.id = df2.id
    """)
    
print(result)

shape: (2, 3)
┌─────┬─────────┬─────┐
│ id  ┆ name    ┆ age │
│ --- ┆ ---     ┆ --- │
│ i64 ┆ str     ┆ i64 │
╞═════╪═════════╪═════╡
│ 2   ┆ Bob     ┆ 25  │
│ 3   ┆ Charlie ┆ 30  │
└─────┴─────────┴─────┘


Общий синтаксис внутреннего соединения имеет следующий вид: `<первая таблица> INNER JOIN <вторая таблица> ON <условие соединения>`.

#### LEFT JOIN (LEFT OUTER JOIN)

Соединение **LEFT JOIN** возвращает все строки из левой таблицы, и совпадающие из правой.
Если в правой таблице нет соответствия, то значения из неё будут null.

Пример:

In [97]:
with pl.SQLContext(df1=df1, df2=df2, eager=True) as ctx:
    result = ctx.execute("""
        SELECT df1.id, df1.name, df2.age
        FROM df1
            LEFT JOIN df2 ON df1.id = df2.id
    """)
print(result)

shape: (3, 3)
┌─────┬─────────┬──────┐
│ id  ┆ name    ┆ age  │
│ --- ┆ ---     ┆ ---  │
│ i64 ┆ str     ┆ i64  │
╞═════╪═════════╪══════╡
│ 1   ┆ Alice   ┆ null │
│ 2   ┆ Bob     ┆ 25   │
│ 3   ┆ Charlie ┆ 30   │
└─────┴─────────┴──────┘


Общий синтаксис левого внешнего соединения имеет следующий вид: `<первая таблица> LEFT JOIN <вторая таблица> ON <условие соединения>`.

#### RIGHT JOIN (RIGHT OUTER JOIN)

Соединение **RIGHT JOIN** возвращает все строки из правой таблицы, и совпадающие из левой.
Если в левой таблице нет соответствия, тозначения из неё будут null.

Пример:

In [98]:
with pl.SQLContext(df1=df1, df2=df2, eager=True) as ctx:
    result = ctx.execute("""
        SELECT df1.id, df1.name, df2.age
        FROM df1
            RIGHT JOIN df2 ON df1.id = df2.id
    """)
print(result)

shape: (3, 3)
┌──────┬─────────┬─────┐
│ id   ┆ name    ┆ age │
│ ---  ┆ ---     ┆ --- │
│ i64  ┆ str     ┆ i64 │
╞══════╪═════════╪═════╡
│ 2    ┆ Bob     ┆ 25  │
│ 3    ┆ Charlie ┆ 30  │
│ null ┆ null    ┆ 35  │
└──────┴─────────┴─────┘


Общий синтаксис правого внешнего соединения имеет следующий вид: `<первая таблица> RIGHT JOIN <вторая таблица> ON <условие соединения>`.

#### FULL JOIN (FULL OUTER JOIN)

Соединение **FULL JOIN** возвращает все строки из обеих таблиц.
Если совпадения нет, то соответствующие столбцы заполняются null.

Пример:

In [99]:
with pl.SQLContext(df1=df1, df2=df2, eager=True) as ctx:
    result = ctx.execute("""
        SELECT df1.id, df1.name, df2.age
        FROM df1
            FULL JOIN df2 ON df1.id = df2.id
    """)
print(result)


shape: (4, 3)
┌──────┬─────────┬──────┐
│ id   ┆ name    ┆ age  │
│ ---  ┆ ---     ┆ ---  │
│ i64  ┆ str     ┆ i64  │
╞══════╪═════════╪══════╡
│ 2    ┆ Bob     ┆ 25   │
│ 3    ┆ Charlie ┆ 30   │
│ null ┆ null    ┆ 35   │
│ 1    ┆ Alice   ┆ null │
└──────┴─────────┴──────┘


Общий синтаксис полного внешнего соединения имеет следующий вид: `<первая таблица> FULL JOIN <вторая таблица> ON <условие соединения>`.

#### CROSS JOIN

Соединение **CROSS JOIN** возвращает декартово произведение строк из обеих таблиц.
Каждая строка из левой таблицы соединяется с каждой строкой из правой.

Пример:

In [100]:
with pl.SQLContext(df1=df1, df2=df2, eager=True) as ctx:
    result = ctx.execute("""
        SELECT df1.name, df2.age
        FROM df1
            CROSS JOIN df2
    """)
print(result)

shape: (9, 2)
┌─────────┬─────┐
│ name    ┆ age │
│ ---     ┆ --- │
│ str     ┆ i64 │
╞═════════╪═════╡
│ Alice   ┆ 25  │
│ Alice   ┆ 30  │
│ Alice   ┆ 35  │
│ Bob     ┆ 25  │
│ Bob     ┆ 30  │
│ Bob     ┆ 35  │
│ Charlie ┆ 25  │
│ Charlie ┆ 30  │
│ Charlie ┆ 35  │
└─────────┴─────┘


Общий синтаксис перекрестного соединения имеет следующий вид: `<первая таблица> CROSS JOIN <вторая таблица>`

#### NATURAL JOIN

Соединение **NATURAL JOIN** автоматически соединяет таблицы по всем столбцам с одинаковыми именами.

In [101]:
df1 = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
    "age": [20, 25, 30]
})

df2 = pl.DataFrame({
    "id": [2, 3, 4],
    "age": [25, 31, 35],
    "city": ["Minsk", "Moscow", "Kiev"]
})

with pl.SQLContext(df1=df1, df2=df2, eager=True) as ctx:
    result = ctx.execute("""
        SELECT *
        FROM df1
            NATURAL INNER JOIN df2
    """)
print(result)

shape: (1, 6)
┌─────┬──────┬─────┬────────┬─────────┬───────┐
│ id  ┆ name ┆ age ┆ id:df2 ┆ age:df2 ┆ city  │
│ --- ┆ ---  ┆ --- ┆ ---    ┆ ---     ┆ ---   │
│ i64 ┆ str  ┆ i64 ┆ i64    ┆ i64     ┆ str   │
╞═════╪══════╪═════╪════════╪═════════╪═══════╡
│ 2   ┆ Bob  ┆ 25  ┆ 2      ┆ 25      ┆ Minsk │
└─────┴──────┴─────┴────────┴─────────┴───────┘


**NATURAL JOIN** есть для INNER, LEFT, RIGHT и FULL соединений. Общий синтаксис: `<первая таблица> NATURAL <INNER/LEFT/RIGHT/FULL> JOIN <вторая таблица>`

#### SEMI JOIN

Соединение **LEFT SEMI JOIN** возвращает только строки из левой таблицы, которые имеют совпадения в правой.
Результат не содержит столбцов из правой таблицы.

Пример:

In [102]:
with pl.SQLContext(df1=df1, df2=df2, eager=True) as ctx:
    result = ctx.execute("""
        SELECT *
        FROM df1
            LEFT SEMI JOIN df2 ON df1.id = df2.id
    """)
print(result)

shape: (2, 3)
┌─────┬─────────┬─────┐
│ id  ┆ name    ┆ age │
│ --- ┆ ---     ┆ --- │
│ i64 ┆ str     ┆ i64 │
╞═════╪═════════╪═════╡
│ 2   ┆ Bob     ┆ 25  │
│ 3   ┆ Charlie ┆ 30  │
└─────┴─────────┴─────┘


**SEMI JOIN** есть для LEFT и RIGHT соединений. Общий синтаксис: `<первая таблица> SEMI <LEFT/RIGHT> JOIN <вторая таблица> ON <условие соединения>`

#### ANTI JOIN

Соединение **LEFT ANTI JOIN** возвращает только строки из левой таблицы, которые не имеют совпадений в правой.
Результат не содержит столбцов из правой таблицы.

Пример:

In [103]:
with pl.SQLContext(df1=df1, df2=df2, eager=True) as ctx:
    result = ctx.execute("""
        SELECT *
        FROM df1
            LEFT ANTI JOIN df2 ON df1.id = df2.id
    """)
print(result)

shape: (1, 3)
┌─────┬───────┬─────┐
│ id  ┆ name  ┆ age │
│ --- ┆ ---   ┆ --- │
│ i64 ┆ str   ┆ i64 │
╞═════╪═══════╪═════╡
│ 1   ┆ Alice ┆ 20  │
└─────┴───────┴─────┘


**ANTI JOIN** есть для LEFT и RIGHT соединений. Общий синтаксис: `<первая таблица> ANTI <LEFT/RIGHT> JOIN <вторая таблица> ON <условие соединения>`

### НОВАЯ ТЕМА

Основное рассказано из того, что поддерживает PolarsSQL.